In [ ]:
import utilities.return_llm_database
import importlib
import torch
importlib.reload(utilities.return_llm_database)

embedding_model, graph_db = utilities.

# 实现相关的实体消融

# 1.neo 4j  Entity Resolver 实体消歧相关概念

首先对实体消歧，进行数学上的定义

$$ M=N,E,E,O,K,\delta $$

其中 $N = \{n_1, n_2, \dots, n_l\}$ 是待消歧的实体名集合（如“李娜”“迈克尔·乔丹”等）。  
$E = \{e_1, e_2, \dots, e_k\}$ 是目标实体列表（如李娜（网球运动员）、迈克尔·乔丹（NBA巨星）等），通常以知识库（如Wikipedia）形式给出。  
$D = \{d_1, d_2, \dots, d_n\}$ 是含待消歧实体名的文档集（如“迈克尔·乔丹”的Google前100搜索结果网页）。  
$O = \{o_1, o_2, \dots, o_m\}$ 是$D$中所有实体指称项集合（如“迈克尔·乔丹是NBA最伟大的球星”中的“迈克尔·乔丹”）。  
$K$ 是消歧任务背景知识，常用目标实体文本描述（如Wikipedia条目），近年扩展至社会化关联、语义关联等知识。

$ \delta: O \times K \to E $  是命名实体消歧函数，用于将待消歧的实体指称项映射到目标实体列表(如果 E 是显式给定的)或者按照其指向的目标实体进行聚类(如果 E 没有显式给定, 是隐藏变量)。命名实体消歧函数是命名实体消歧任务的核心部分，直接影响系统的性能。

----

实体对齐可以分为传统方法和基于表示学习的方法，

其中传统方法又可以分为基于等价关系推理和基于相似度的方法，图方法。

* 基于等价推理的方法是一种基于符号推理的方法，其主要基于关联数据中的owl:sameAs 进行等价关系推理。等价映射声明了概念之间和关系之间的对应，异构本体的等价成分之间

在互操作过程中可以直接相互代替。

* 基于相似度的计算核心思想是通过计算实体的特征计算实体的相似度。包括标签，名称，特征等，但总体的效果一般。

----

* 基于聚类的实体对齐： 所有指向同一个目标的实体的指称项被消歧系统聚在同一个类别中，聚类结果的每一个类别对应一个目标的实体。
> 实际上，基于相似度的计算可以认为是聚类实体对齐的前导步骤，在高维基础上，通过聚类找到相似点，然后形成合并。

在目标实体没有给定的情况下，目前绝大多数系统都采用聚类方法进行命名实体消歧。给定待消歧的实体指称项集合 $O = \{o_1, o_2, \cdots, o_k\}$，以聚类方式实现消歧的系统按如下步骤进行消歧。

(1) 对每一个实体指称项 $o$，抽取其特征（如上下文中的词、实体、概念），并将其表示成特征向量 $o = (w_1, w_2, \cdots, w_n)$。

(2) 计算实体指称项之间的相似度。

(3) 采用某种聚类算法对实体指称项聚类，使得聚类结果中每一个类别都对应到一个目标实体上。

以聚类方式实现实体消歧的关键问题是计算指称项之间的相似度。按照实体指称项相似度计算方法的不同，基于聚类的实体消歧系统可以分为如下三类：

  1. 基于表层特征的实体指称项相似度计算；
     可以理解为基于embedding后的高纬度相似度的聚类。
   
  2. 基于扩展特征的实体指称项相似度计算；
     应该可以理解为对表层特征的一种补充，也是和本体知识可以结合较为紧密的一种方法，其核心是使用知识资源（例如wiki，例如本地rdf知识），来扩充实体指称项的特征表示。

     对基于embedding 的聚类结果进行重构。例如如果两个指称项指向同一个rdf知识（owl:sameas)，那么就直接考虑其合并，而不是基于embedding。

     即通过分类体系（Taxonomy）来进行对齐。
   
  3. 基于社会化网络的实体指称项相似度计算。
     多基于图算法，利用社会化关系的传递性，考虑隐藏的关系知识，即社会网络更为接近的实体认为合并的可能性更强，

     一般使用随机游走算法进行计算。



基于实体链接的实体对齐方法

基于实体链接的实体消歧方法，一般是将实体指称项链接到知识库中特定的实体，也称为实体链接。实体链接（通常称为 Entity Linking，与 Entity Grounding，Entity Resolution，Record Linkage 和 Entity Disambiguation 意义接近）指的是将一个命名实体的文本指称项（Textual Mention）链接到知识库中相应实体的过程（知识库中可能不包含待消歧指称项的对应实体，这时，将实体指称项链接到空实体 NIL）。

通常，实体链接的输入包括两个部分。

（1）目标实体知识库：目前最常用的知识库为 Wikipedia，在其他一些任务中也可能是特定领域的知识库，如电影领域的 IMDB、科学研究领域的 DBLP 等。知识库通常包含如下的一些信息：
实体表、实体的文本描述、实体的结构化信息（如属性/属性值对）、实体的辅助性信息（如实体类别）；同时知识库也经常提供一些额外的结构化语义信息，如实体之间的关联。

这里是否可以用agent来实现，也是一个研究点，或者创新点？

（2）待消歧实体指称项及其上下文信息。

实体链接任务通常需要两个步骤。

实体链接通常包含如下两个步骤。

（a）**链接候选过滤（Blocking）**：由于一个知识库中通常包含上百万个实体，在实际的实体链接任务中不可能计算一个指称项与所有实体之间进行链接的可能性。因此，实体链接需要首先根据规则或知识过滤掉大部分该指称项不可能指向的实体，仅仅保留少量链接实体候选。

（b）**实体链接（Linking）**：给定指称项及其链接候选，实体链接的第二个步骤是确定该实体指称项最终指向的目标实体。

目前，大部分实体链接研究的重点在于第二步，即如何根据实体指称项的上下文信息和知识库中实体的信息从链接候选中确定目标实体。



实际上，研究中，采取的是聚类与链接结合的方法，

其中链接方法对应的知识库，即本地的rdf生物实验，obi等语义文件，以及结合wikipedia的联网知识库。

并可以通过加权打分的方式，来判断实体之间的接近程度。



The KG Writer component creates new nodes for each identified entity without making assumptions about entity similarity. 

The Entity Resolver is responsible for refining the created knowledge graph by merging entity nodes that represent the same real-world object.

In practice, this package implements three resolvers:

* a simple resolver that merges nodes with the same label and identical “name” property;
> 这里是根据name属性来进行合并，需要注意的是，neo4j GR 的方案中，节点name实际起到了text的作用，故而这种合并理论上只对独立概念，专有名词等会由于merge的作用

* two similarity-based resolvers that merge nodes with the same label and similar set of textual properties (by default they use the “name” property):

   * a semantic match resolver, which is based on spaCy embeddings and cosine similarities of embedding vectors. This resolver is ideal for higher quality KG resolution using static embeddings.
     > 通过Spacy 实现name的embdding ，并基于余弦相似度实现合并。

   * a fuzzy match resolver, which is based on RapidFuzz for Rapid fuzzy string matching using the Levenshtein Distance. This resolver offers faster ingestion speeds by using string similarity measures, at the potential cost of resolution precision.
* <font color='red'>neo4j 的embedding 余弦合并方案与spacy绑定，其embdding向量也需要使用spacy的内置向量，这种方式实际很不方便，且效率很低</font>

* 应该在neo4j中就为各个节点生成embedding，同时通过自定义的merge 来进行合并 *

## 1.2.neo4j 官方 merging 概念，方法

neo4j GR的方案merging 方案所使用的算法差别很大，一个是基于embedding 余弦相似度，一个是基于字符相似度的替换算法，总结如下：

下面是对 **RapidFuzz** 的介绍，以及它与 **embedding 余弦相似度** 的本质区别分析：

---

#### 1.2.1  RapidFuzz  

**RapidFuzz** 是一个基于 C++ 的高性能 Python 库，用于快速模糊字符串匹配和相似度计算。它是 [fuzzywuzzy](https://github.com/seatgeek/fuzzywuzzy) 的现代化替代版本，**速度更快、内存更省、不依赖 Python 的 slow string handling**。

##### (1) 特点

* 支持多种字符串相似度算法：

  * **Levenshtein 距离**
  * **Jaro / Jaro-Winkler 距离**
  * **Token sort ratio / Token set ratio**（对无序文本有效）
* 无需预训练或大模型，即可快速用于实体比对
* 内置多种“模糊匹配评分”方法，比如：

  * `fuzz.ratio()`
  * `fuzz.partial_ratio()`
  * `fuzz.token_sort_ratio()` 等

---

##### (2)fuzz 与 Embedding + Cosine Similarity 的区别

| 比较维度          | RapidFuzz（如 Levenshtein）  | Embedding + Cosine Similarity                   |
| ------------- | ------------------------- | ----------------------------------------------- |
| **输入**        | 字符串本身                     | 向量（通常是通过语言模型生成的）                                |
| **技术核心**      | 编辑距离（字符级别操作）              | 向量空间几何关系                                        |
| **是否语义感知**    | ❌ 不具备                     | ✅ 具备（例如 “rice” ≈ “white rice”）                  |
| **性能**        | 快速，适用于小数据匹配               | 向量计算较重，但适用于大规模语义比较                              |
| **适合的场景**     | 拼写纠错、命名匹配、实体歧义解决          | 知识检索、语义相似度比较、LLM语义匹配                            |
| **是否需要训练/模型** | ❌ 不需要                     | ✅ 需要 Embedding 模型（如 BERT, BCE）                  |
| **示例库**       | `rapidfuzz`, `fuzzywuzzy` | `SentenceTransformers`, `Huggingface`, `OpenAI` |

---

##### (3) 示例比较
 

```python
from rapidfuzz import fuzz

fuzz.ratio("rice samples", "rice sample")  # 输出约 96（字符相似）
fuzz.ratio("rice", "white rice")           # 输出约 67（字符不完全重合）
```

#### 1.2.2  Embedding + Cosine 相似度：

```python
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')
emb1 = model.encode("rice")
emb2 = model.encode("white rice")

similarity = util.cos_sim(emb1, emb2)  # 输出约 0.92（语义相近）
```

---

####  总结：使用建议

| 场景需求              | 推荐技术                 |
| ----------------- | -------------------- |
| 快速匹配字符串（如实体ID/名称） | ✅ RapidFuzz          |
| 语义理解、同义词处理        | ✅ Embedding + Cosine |
| 知识图谱实体消歧（字符级别）    | ✅ RapidFuzz          |
| 多语言语义消歧或跨领域匹配     | ✅ Embedding          |

---

如果构建一个结合 Neo4j Graph RAG 的实体解析系统，可以：

* **初步用 RapidFuzz 去重合并（高效）**
* **高级使用嵌入 + cosine similarity 做语义级精化**
 

-----

以下的blog中也实现了通过embedding的Entitiy merge 但其方法可能已经融合到官方方案之中。
[blog 方案](https://neo4j.com/blog/developer/property-graph-index-llamaindex/)
 


#### 1.3 neo4j 官方给出两种直接可以用的融合方法

> 基于字符匹配的融合
  > 基于字符的融合又分为高精度融合，以及低精度融合
  *  SinglePropertyExactMatchResolver
  
> 基于spaCy 词向量 的融合方法
  *SpaCySemanticMatchResolver

第一种方法比较简单，适合概念上的融合，即由于概念名称的相似性可以先用SinglePropertyExtraMatchResolver 进行过滤

```python
from neo4j_graphrag.experimental.components.resolver import (
    SinglePropertyExactMatchResolver,
    SpaCySemanticMatchResolver,
    FuzzyMatchResolver,
)

 

# 选用一种 resolver（你当前用的是 spaCy 语义匹配）
resolver = SpaCySemanticMatchResolver(neo4j_driver)

# 运行实体消歧 / 去重
res = await resolver.run()
print(res)
```

* 这些 **resolver** 属于 *KG Builder* 的“实体解析（去重/合并）”步骤：在抽取后把指向同一真实实体的多个节点合并起来，减少重复，清洁图数据。
* `SpaCySemanticMatchResolver` 会基于 spaCy 的词向量对节点文本属性做**语义相似**匹配，认为相似就合并。(以下被自定义方法取代)
* `SinglePropertyExactMatchResolver` 做**单属性精确匹配**（默认是 `name`），完全相同才合并。
* `await resolver.run()` 会在图上真正执行合并操作，并返回一次运行的统计信息（合并了多少、跳过多少等）。

> 官方文档把这些能力放在“KG Builder → Entity Resolver”，并且在 API 文档里列出了这些 resolver 类。([Graph Database & Analytics][1])

---

 

##### (1) SinglePropertyExactMatchResolver

* **作用**：按某个**单一属性的精确值**匹配（默认使用 `name`）来判定“是否为同一实体”，相同则合并。适用于你已经有**规范化命名**的场景。([Graph Database & Analytics][1])
* **范围控制**：可以用 `filter_query` 将匹配范围限制在某些节点（例如只处理带 `__Entity__` 的实体节点，或只处理某个子图），避免把不相关节点放进候选集合里。([Graph Database & Analytics][1])
* **适用场景**：法定名称、唯一编号、标准代码等字段稳定且一致时，效果最好。
* **注意**：默认看 `name` 属性；如果你的实体名存在于其他字段（比如你图里常用 `WHU_HASNAME`），建议在构建阶段把它同步/映射到 `name`，再用该 resolver 做精确去重。([Graph Database & Analytics][1])

> 文档在“KG Builder 用户指南 → Entity Resolver”中明确了“精确匹配按 name 属性”、“可用 filter\_query 限定范围”等；API 页里也能看到该类的定义。([Graph Database & Analytics][1])

##### (2)SpaCySemanticMatchResolver

* **作用**：用 **spaCy 词向量**对节点的文本属性做**语义相似**匹配（不是简单字符串相等），用于合并“名称相近/同义”的重复实体。([Graph Database & Analytics][1])
* **依赖**：需要安装 `spacy` 以及合适的语言模型（例如英文常用 `en_core_web_lg`）。官方安装说明把它列为 *optional dependency*（“nlp”），专门用于该 resolver。([Graph Database & Analytics][1])
* **模型选择**：效果取决于所选的 spaCy 模型与语言覆盖；不同语言（或领域词汇）需要相应模型，语义相似阈值也要按你的数据分布微调。([Graph Database & Analytics][1])
* **范围控制**：同样可以用 `filter_query` 只在某些节点集合里做匹配，控制候选集规模与成本。([Graph Database & Analytics][1])
* **适用场景**：名字有变体/缩写/同义（如“IBM” vs “International Business Machines”）或数据存在拼写差异时，比精确匹配更鲁棒。

> 官方文档明确这是一种语义匹配的实体解析器，并给出依赖安装与使用提示；Release notes 也记录了它的新增与改进。([Graph Database & Analytics][1])

---



* 你当前直接 `SpaCySemanticMatchResolver(neo4j_driver)` 使用默认设置，通常会：

  1. 选取目标节点集合（可用 `filter_query` 缩小范围，例如只对 `:__Entity__` 或你关心的标签运行），
  2. 基于 spaCy 向量计算候选对的相似度，
  3. 超过阈值就**合并节点**（以及相应关系），返回本次解析的统计信息。([Graph Database & Analytics][1])
* 如果你的实体名称字段主要是 `WHU_HASNAME`，而不是 `name`：

  * 对**精确匹配**路线，建议在建图时把 `WHU_HASNAME` 同步到 `name`；
  * 对**语义匹配**路线，也建议统一一个用于匹配的文本字段（通常也是 `name`），这样与文档默认行为一致，配置和维护都更省心。([Graph Database & Analytics][1])

---



1. **先小范围 dry-run**：用 `filter_query` 只在一小撮节点上跑，确认阈值与合并效果再扩大范围，避免误合并。([Graph Database & Analytics][1])
2. **安装依赖**（仅 spaCy 方案）：`pip install spacy` 并下载合适模型（如 `python -m spacy download en_core_web_lg`）。官方把这列为可选依赖“nlp”。([Graph Database & Analytics][1])
3. **性能与成本**：这些 resolver 不走 LLM API 调用，spaCy/rapidfuzz 都是本地库，速度快、成本低；但候选对过多仍会慢，所以一定要**限制范围**。([Graph Database & Analytics][1])

---



* Neo4j GraphRAG Python – *User Guide: Knowledge Graph Builder*（包含 Entity Resolver 设计、行为与用法说明）. ([Graph Database & Analytics][1])
* Neo4j GraphRAG Python – *API Documentation*（列出 `SinglePropertyExactMatchResolver`、`SpaCySemanticMatchResolver` 等类）. ([Graph Database & Analytics][2])
* Release Notes：新增 spaCy 语义匹配 resolver 的版本说明与更新点。([Graph Database & Analytics][2])



[1]: https://neo4j.com/docs/neo4j-graphrag-python/current/user_guide_kg_builder.html "User Guide: Knowledge Graph Builder — neo4j-graphrag-python  documentation"
[2]: https://neo4j.com/docs/neo4j-graphrag-python/current/api.html "API Documentation — neo4j-graphrag-python  documentation"


In [12]:
from rapidfuzz import fuzz

print(fuzz.ratio("rice biomass", "rice grains (brown rice)") ) # 输出字符相似度
print(fuzz.ratio("brown rice", "white rice") )          #  输出字符相似度

38.888888888888886
60.0


In [ ]:
from sentence_transformers import SentenceTransformer, util

#model = SentenceTransformer('all-MiniLM-L6-v2')
emb1 = embed_model.encode("rice biomass")
emb2 = embed_model.encode("rice grains (brown rice)")
 
similarity = util.cos_sim(emb1, emb2)  # 输出约 0.92（语义相近）
 

### 实验
 kg 中实体合并一般适用于受控词表的合并，这里通过specimen进行验证，合并前 speceimen count =21

在此实现pipeline方式，首先通过cyper 链接数据库，得到目前可以融合的实体

1. 先通过cyper 链接数据库，得到目前可以融合的实体
   1.1 实现完全对应的实体的融合


2. 首先实现single property 精确匹配
3. 实现fussy的匹配，并建立master entity作为中介节点
4. 实现向量化的匹配
5. 实现基于agent+llm得到的共同节点的匹配
 


1. 精准查询得到的可融合节点

In [1]:
import utilities.return_llm_database
import pandas as pd

r = utilities.return_llm_database.DatabaseManager()
llm, embed_model, neo4j_driver = r.get_components()
        
def get_mergeable_nodes(property_name="WHU_HASNAME"):
    """返回可融合节点的DataFrame"""
    
    query = f"""
    MATCH (e:__Entity__)
    WITH labels(e) AS labs, 
         toLower(trim(toString(e.`{property_name}`))) AS key, 
         collect(elementId(e)) AS ids
    WHERE key IS NOT NULL AND key <> '' AND size(ids) > 1
    RETURN labs, key, size(ids) AS cnt, ids
    ORDER BY cnt DESC
    """
    
    with neo4j_driver.session() as session:
        try:
            result = session.run(query)
            data = []
            
            for record in result:
                data.append({
                    'labels': record['labs'],
                    'key_value': record['key'], 
                    'node_count': record['cnt'],
                    'entity_ids': record['ids']
                })
            
            df = pd.DataFrame(data)
            print(f"找到 {len(df)} 组可融合节点，共 {df['node_count'].sum() if len(df) > 0 else 0} 个重复节点")
            return df
            
        except Exception as e:
            print(f"查询失败: {e}")
            return pd.DataFrame()

def check_all_properties():
    """检查所有WHU属性的可融合情况"""
    
    properties = ["WHU_HASNAME", "WHU_HASORIGINALTEXT"]
    
    for prop in properties:
        print(f"\n--- {prop} ---")
        df = get_mergeable_nodes(prop)
        if not df.empty:
            print(df[['key_value', 'node_count']].head())
    
    return df

# 直接执行
if __name__ == "__main__":
    df = check_all_properties()
    print(f"\n完成！返回DataFrame包含: {list(df.columns) if not df.empty else '无数据'}")

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /maidalun1020/bce-embedding-base_v1/resolve/main/modules.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001D34FD31650>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 048ac66a-3757-44a0-bcf8-da198d7d0986)')' thrown while requesting HEAD https://huggingface.co/maidalun1020/bce-embedding-base_v1/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /maidalun1020/bce-embedding-base_v1/resolve/main/modules.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001D34FD32990>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 2d68cf49-3ac3-4ef4-b753-7ef209b1b5db)')' thrown while requesting HEAD https://huggingface.co/maidalun1020/bce-embedding-base_v1/re

KeyboardInterrupt: 

# pipeline code block A 通过SinglePropertyExactMatchResolver 实现高精度字符串匹配消歧



1. 通过WHU_HASORIGINALTEXT（原始文本）
2. 直接合并，不建立master entity
3. 不合并所有节点，只考虑独立概念节点，但其中实际使用single 方式节点包括
   （1） 

  * whu_Target_analyte 化学对象
  * whu_Reagen 化学品
  * whu_Specimen 样品
  * envo_Material 物质

  
   而
   * whu_EnvironmentFeature 环境；* whu_Device 设备 由于带有属性，例如不同版本的Device，同名而不同的地理位置等，不通过名称匹配合并
   * mp_Attribution 贡献者（作者，机构） 通过直接合并可能会产生大量的同名异议问题，故也不直接合并，而是通过master entity节点方式，可以考虑通过fuzzy匹配，然后指向共同的master entity节点方式
   * *mp_References 引用 考虑通过agent，找到odi，进行高精度合并
其特点如下：
 

1. **完全匹配（Exact match）**
   `SinglePropertyExactMatchResolver` 的语义就是：对\*\*同一标签（label）**的节点，按某一个属性（默认是 `name`）的“完全相等”** 来判定可合并（或对齐）。官方源码文档写明：*“Resolve entities with same label and exact same property (default is 'name')”*。([Graph Database & Analytics][1])

2. **仅在相同类型（同一标签）的节点内匹配**
   该 resolver 只在**标签相同**的节点之间做匹配；因此 `Person(name='tom')` 与 `City(name='tom')` 虽然名字相同，但**标签不同**，不会被合并。用户指南也将它作为一种简单的属性匹配式 resolver，与其他（如 SpaCy/Fuzzy）并列说明，用于替换（对齐）由 KG writer 创建的候选节点。([Graph Database & Analytics][1])

补充：

* 可以在创建该 resolver 时**指定属性名**（不一定是 `name`），但依旧是**单字段的“全等匹配”**与**同标签约束**。([Graph Database & Analytics][1])
 

[1]: https://neo4j.com/docs/neo4j-graphrag-python/current/_modules/neo4j_graphrag/experimental/components/resolver.html?utm_source=chatgpt.com "neo4j_graphrag.experimental.components.resolver"


In [2]:
# 实现概念类节点的名称完全匹配消融
from neo4j_graphrag.experimental.components.resolver import (
    SinglePropertyExactMatchResolver,
    SpaCySemanticMatchResolver,
    FuzzyMatchResolver,
)

import utilities.return_llm_database 
from neo4j import GraphDatabase
from utilities import return_llm_database as r
 


from neo4j_graphrag.experimental.components.resolver import SinglePropertyExactMatchResolver
# 获取 Neo4j 驱动（本地数据库）
neo4j_driver = r.DatabaseManager.get_neo4j_driver(remotedatebase=False)

#manager =utilities.return_llm_database.DatabaseManager()
#_,_ ,neo4j_driver=manager.get_components()

filter_q = (
    "WHERE any(l IN labels(entity) WHERE l IN "
    "['whu_Target_analyte','whu_Reagent','whu_Specimen','envo_Material']) "
    "AND coalesce(entity.WHU_HASORIGINALTEXT, '') <> ''"
)

exact = SinglePropertyExactMatchResolver(
    driver=neo4j_driver,
    filter_query=filter_q,
    resolve_property="WHU_HASORIGINALTEXT",
    neo4j_database="neo4j",
)
await exact.run()


#filter_query="""MATCH (n:__Entity__)
      #WHERE exists(n.WHU_HASNAME) OR exists(n.WHU_HASORIGINALTEXT)
     # // 这里可加 NOT elementId(n) IN $exact_ids 之类的条件，跳过已处理
     # RETURN n"""
# 过滤掉 exact 已经判重过的节点（可通过 elementId 集合差集实现）
#fuzzy = await FuzzyMatchResolver(
#    neo4j_driver,
#    resolve_properties=["WHU_HASNAME", "WHU_HASORIGINALTEXT"],
#    similarity_threshold=0.90,
    #filter_query=filter_query,
    # ).run()

print("single exact match result:",exact)
print("##################################################")
#print("fuzzy match result:",fuzzy)

single exact match result: <neo4j_graphrag.experimental.components.resolver.SinglePropertyExactMatchResolver object at 0x000001D3C8213590>
##################################################


# pipeline code block B 基于fuzzy相似度


对于其他独立概念节点，采取不同的合并策略，包括通过agent结合llm进行合并，以及在此通过fuzzy匹配进行合并。

fuzzy 是通过字符串模糊匹配算法实现对齐，在此使用apoc.text.jaroWinklerDistance 实现。算法为 

Jaro-Winkler 距离是一种用于衡量两个字符串相似度的算法，特别擅长处理短字符串（如人名、地址）的模糊匹配问题。它的核心思想是，不仅考虑字符匹配的数量和顺序，还特别奖励那些具有相同前缀的字符串，这使得它在数据清洗、拼写检查和记录去重等场景中非常实用。

下表概述了该算法的核心组成部分和特点：

特性 描述

算法基础 基于 Jaro 相似度，并在此基础上增加了对公共前缀的奖励机制。

核心思想 相似度与匹配字符的数量成正比，同时考虑字符顺序（换位数）的影响。对字符串开头的匹配给予额外权重。

输出范围 得分介于 0 到 1 之间。1 表示完全匹配，0 表示完全不同。

独特优势 对前缀相似的字符串更敏感，例如比较“Samuel”和“Sam”时，得分会高于其他相似度算法。

⚙️ 算法原理分步解析

Jaro-Winkler 距离的计算分为两个主要阶段：

1. 计算 Jaro 相似度 (simj)
首先，算法会计算两个字符串的 Jaro 相似度，公式如下：

$$
simj = \frac{1}{3} \left( \frac{m}{s1 } + \frac{m}{ s2 } + \frac{m - t}{m} \right
$$

其中：
•  s1  和  s2
  分别是两个字符串的长度。

•  m  是匹配的字符数。一个字符被认为是匹配的，必须满足两个条件：(1) 字符相同；(2) 它在另一个字符串中的位置不能超过预设的匹配窗口（通常为 
$$
\left\lfloor \frac{\max(s1 , s2
)}{2} \right\rfloor - 1 \)）
$$

•  t  是换位数目的一半。换位指的是匹配的字符在两个字符串中顺序不一致的情况。

1. 应用 Winkler 改进（前缀奖励）
在 Jaro 相似度的基础上，Winkler 改进通过一个简单的公式提升具有共同前缀的字符串的分数：
$$
simw = simj + (l \cdot p \cdot (1 - simj))
$$
其中：
•  simj  是第一步计算出的 Jaro 相似度。

•  l  是两个字符串的公共前缀长度，通常最大只考虑前 4 个字符。

•  p  是缩放因子（通常取值为 0.1），用于控制前缀奖励的强度。这个值不能超过 0.25，否则可能导致最终得分超过 1。

📊 典型应用场景

得益于其对前缀的敏感性，Jaro-Winkler 距离在以下领域表现出色：
• 数据清洗与去重：识别并合并数据库中拼写略有不同的记录，如客户姓名（“John Smith” vs “Jon Smith”）或公司名称。

• 记录链接：在缺乏唯一标识符的情况下，链接来自不同数据源的记录，例如通过人名或地名进行关联。

• 拼写检查与纠错：为输入的错误单词从候选词列表中找出最可能的正确单词。

⚠️ 优势与局限

• 优势：特别适合处理短文本和注重开头准确性的场景，计算效率较高。

• 局限：对于后缀不同但前缀相同的字符串（如“appropriate”和“appropriation”），相似度会很高，但这不一定总是用户想要的。同时，对于前缀不同但整体相似的字符串（如“abcdfox”和“fox”），得分可能会非常低，因为算法对前缀的依赖很强。

 


Jaro-Winkler 距离算法（建议在使用之前进行大小写转换）

基于该算法，进行独立概念节点的匹配节点类包括
 
   * mp_Attribution 贡献者（作者，机构） 通过直接合并可能会产生大量的同名异议问题，故也不直接合并，而是通过master entity节点方式，可以考虑通过fuzzy匹配，然后指向共同的master entity节点方式

----

同时基于 neo4j 书籍，其工作流程包括：
1. 通过jw算法建立similary关联
   根据jw算法，neo4j中存在大量满足jw距离小于0.2的节点对，这些节点对可以作为候选节点对，进行合并。(注意大小写统一问题)

```cypher 
MATCH (p1:mp_Attribution), (p2:mp_Attribution)
WHERE NOT (p1)-[:SAME_AS]-(p2) 
  AND id(p1) > id(p2)
  AND apoc.text.jaroWinklerDistance(
        toLower(p1.WHU_HASORIGINALTEXT), 
        toLower(p2.WHU_HASORIGINALTEXT)
      ) < 0.2
RETURN p1, p2
```

在此基础上，创建SIMILAR关系

``` cypher
CREATE (p1)-[:SIMILAR { sim_score : 1 -
 apoc.text.jaroWinklerDistance(p1.WHU_HASORIGINALTEXT, p2.WHU_HASORIGINALTEXT)}]->(p2)
```
2. 通过wcc算法构建master entity
3. 建立master entity 到各个节点之间的reference关系


## B1 基于fuzz相似度，建立节点间 similar 关系

In [3]:
import utilities.return_llm_database  as rldb
import importlib # 如果尚未导入importlib，需要先导入
importlib.reload(rldb)
 
driver = rldb.DatabaseManager.get_neo4j_driver(remotedatebase=False)

MAX_DIST = 0.2  # 与你示例一致：distance 越小越相似

QUERY_CREATE_SIMILAR = """
/*
为 mp_Attribution 节点对建立 :SIMILAR 关系：
- 排除已有 :SAME_AS 的节点对
- 使用 APOC jaro-winkler 距离做阈值过滤（< $max_dist）
- 避免重复与自环，按 id(p1) > id(p2) 只取一侧
- 仅在首次创建时写 sim_score 与时间戳
*/
CALL {
  MATCH (p1:mp_Attribution), (p2:mp_Attribution)
  WHERE id(p1) > id(p2)
    AND NOT (p1)-[:SAME_AS]-(p2)
    AND coalesce(p1.WHU_HASORIGINALTEXT, '') <> ''
    AND coalesce(p2.WHU_HASORIGINALTEXT, '') <> ''
  WITH p1, p2,
       apoc.text.jaroWinklerDistance(
         toLower(p1.WHU_HASORIGINALTEXT),
         toLower(p2.WHU_HASORIGINALTEXT)
       ) AS dist
  WHERE dist < $max_dist
  MERGE (p1)-[r:SIMILAR]->(p2)
  ON CREATE SET
    r.sim_score = 1.0 - dist,
    r.createdAt = datetime()
  RETURN count(r) AS created_count
}
RETURN created_count
"""

with driver.session() as s:
    rec = s.run(QUERY_CREATE_SIMILAR, {"max_dist": MAX_DIST}).single()
    print("SIMILAR relationships created:", rec["created_count"])

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL () { ... }} {position: line: 9, column: 1, offset: 167} for query: "\n/*\n为 mp_Attribution 节点对建立 :SIMILAR 关系：\n- 排除已有 :SAME_AS 的节点对\n- 使用 APOC jaro-winkler 距离做阈值过滤（< $max_dist）\n- 避免重复与自环，按 id(p1) > id(p2) 只取一侧\n- 仅在首次创建时写 sim_score 与时间戳\n*/\nCALL {\n  MATCH (p1:mp_Attribution), (p2:mp_Attribution)\n  WHERE id(p1) > id(p2)\n    AND NOT (p1)-[:SAME_AS]-(p2)\n    AND coalesce(p1.WHU_HASORIGINALTEXT, '') <> ''\n    AND coalesce(p2.WHU_HASORIGINALTEXT, '') <> ''\n  WITH p1, p2,\n       apoc.text.jaroWinklerDistance(\n         toLower(p1.WHU_HASORIGINALTEXT),\n         toLower(p2.WHU_HASORIGINALTEXT)\n       ) AS dist\n  WHERE dist < $max_dist\n  MERGE (p1)-[r:SIMILAR]->(p2)\n  ON 

SIMILAR relationships created: 1851


## B2 在similar关系基础上，基于WCC 算法构成对齐集合，并建立 master entity 中介节点与reference关系
在以上similar 关系的基础上，通过wcc算法，将similar 关系的节点分为多个集合，每个集合中的节点都是等价的实体。

All nodes in a set form a connected component and are equivalent to a unique entity.

Detecting such sets is what the Weakly Connected Components (WCC) algorithm

does. The adjective weakly comes from the fact that the algorithm works on undirected graphs.

This nicely fits the objective of this exercise because the direction of both

SAME_AS and SIMILAR relationships is irrelevant, as both are symmetric.

----

以下代码中，涉及neo4j GDS 的几个程序功能点，包括
1. project
2. stream，write，mutate，state等

project 相关资料 https://neo4j.com/docs/graph-data-science/current/management-ops/graph-creation/graph-project/

其可以理解为一种graph view， 即内存中的虚拟视图，不对原graph 进行修改，只是在内存中创建一个视图，用于执行gds 算法。

而对应 stream，write，mutate，state 等操作， 是与project 相关的执行模式：·


在Neo4j的图数据科学库（GDS）中，gds.wcc.stream 的 stream 是一种执行模式，它表示算法（在这里是弱连通分量算法WCC）的计算结果将作为数据流直接返回给用户，而不会修改原始的图数据或将其写入数据库。

GDS库中几种核心执行模式的区别，见下表：

执行模式 关键字 作用 是否修改图数据

| 模式 | 描述 | 是否写入数据库 |
|------|------|----------------|
| 流模式 (stream) | 将结果（如节点ID及其对应的组件ID）作为数据流返回，供即时分析或进一步处理 | 否 |
| 统计模式 (stats) | 返回一条包含算法运行汇总信息的记录，如找到的连通分量总数 | 否 |
| 变更模式 (mutate) | 将算法的结果（如组件ID）作为一个新属性写入到内存中的图投影里，便于后续算法使用 | 仅修改内存中的图 |
| 写入模式 (write) | 将算法的结果（如组件ID）作为属性持久化地写回到Neo4j数据库中的原始节点上 | 是，写入磁盘数据库 |

💡 为何使用 stream 模式？

选择 stream 模式通常基于以下考虑：
•   即时查看结果：当你希望快速查看算法结果，进行探索性数据分析或验证算法效果时，stream 模式是最直接的选择。它让你能立即看到每个节点被分配到了哪个连通分量（componentId）。

•   数据不持久化：如果你的分析流程不需要将中间结果（如社区ID）保存下来，或者你只想将最终筛选出的重要结果写回数据库，那么使用 stream 模式可以避免产生不必要的数据冗余。

•   与其他查询结合：返回的数据流可以很方便地使用 Cypher 语句进行进一步的过滤、排序或聚合。例如，你可能只关心包含节点数量大于某个阈值的连通分量。

🛠️ 典型用法示例

一个典型的使用 gds.wcc.stream 模式的查询如下所示：
```cypher
CALL gds.wcc.stream('your_graph_name')
YIELD nodeId, componentId
RETURN gds.util.asNode(nodeId).id AS nodeName, componentId
ORDER BY componentId, nodeName
```

在这个查询中：
•   CALL gds.wcc.stream('your_graph_name') 调用了WCC算法在指定图投影上以流模式运行。

•   YIELD nodeId, componentId 指定算法返回的字段，即节点的内部ID和它所属的连通分量ID。

•   RETURN gds.util.asNode(nodeId).id AS nodeName, componentId 将内部节点ID转换回实际的节点对象，并获取其有意义的属性（如id或name）用于展示。

💎 总结

简单来说，gds.wcc.stream 中的 stream 就是一个“只读”模式。它专注于计算并将结果立刻呈现给你，非常适合进行数据探索和快速验证。当你需要将分析结果保存下来用于后续操作或可视化时，才会考虑使用 write 或 mutate 模式。

此外：​资源管理​：图投影会占用大量内存。使用后，如果确定不再需要，应使用 CALL gds.graph.drop('graph-name')命令将其从内存中删除，以释放资源

-----


1. 首先通过 project 构建一个包含mp_Attribution 节点的图投影
```cypher
CALL gds.graph.project(
 'identity-wcc',
 'mp_Attribution',
 ['SIMILAR']
)
```

1. 再在此project基础上，通过stream执行wcc算法，将similar 关系的节点分为多个集合，每个集合中的节点都是等价的实体。
```cypher
CALL gds.wcc.stream('identity-wcc')
YIELD nodeId, componentId
RETURN gds.util.asNode(nodeId).id AS nodeName, componentId
ORDER BY componentId, nodeName
```
1. 在构建的identity-wcc 视图基础上，首先构建pg:AttributionMaster 节点，实现构建master enttiy，而后建立(pg)-[:HAS_REFERENCE]->(mp_Attribution) 引用链接
```cypher

CALL gds.wcc.stream('identity-wcc')
YIELD nodeId, componentId
WITH gds.util.asNode(nodeId) AS mp_Attribution, componentId AS golden_id
// 首先，MERGE AttributionMaster 节点，并根据是否创建设置属性
MERGE (pg:AttributionMaster { uid: golden_id })
ON CREATE SET pg.originalText = mp_Attribution.WHU_HASORIGINALTEXT, pg.hasName = mp_Attribution.WHU_HASNAME
// 然后，在已确定的两个节点之间MERGE关系
WITH pg, mp_Attribution
MERGE (pg)-[:HAS_REFERENCE]->(mp_Attribution)
// 可以返回一些信息以确认操作结果，例如：
RETURN pg.uid AS masterUID, mp_Attribution.WHU_HASNAME AS attributionName
```

4. 通过结果观察，大部分Attribution节点构成有意义的 master entity 对应关联合并，但有部分master enity节点reference节点过多，观察发现可能原因在这些节点文本过于简单，造成之间过于相似。

解决方法为直接去掉出度大于10 的master entity 节点

首先找到出度大于10 的 pg:AttributionMaster 极其关系
```cypher
MATCH (startNode)-[r:HAS_REFERENCE]->(endNode)
WITH startNode, count(r) AS connectionCount
WHERE connectionCount > 10
RETURN startNode, connectionCount
```
基于以上结果，直接删除出度大于10的 pg:AttributionMaster 节点及其关系（这里直接用 Count计算）
```cypher
MATCH (pg:AttributionMaster)
WHERE COUNT { (pg)-[:HAS_REFERENCE]->() } > 10
DETACH DELETE pg
```

In [4]:
import utilities.return_llm_database as rldb
import importlib; importlib.reload(rldb)
llm = rldb.DatabaseManager.get_llm()

In [7]:
! pip install --upgrade torch>=2.6

^C


In [9]:
# 对应python 脚本：


import utilities.return_llm_database as rldb
import importlib; importlib.reload(rldb)
driver = rldb.DatabaseManager.get_neo4j_driver(remotedatebase=False)

GRAPH_NAME = "identity-wcc"
MAX_OUT_DEG = 10

# -- Step 0: 若存在同名图投影，尝试丢弃（防止重复创建失败）
DROP_GRAPH = """
CALL gds.graph.drop($name, false) YIELD graphName
RETURN graphName
"""
try:
    with driver.session() as s:
        s.run(DROP_GRAPH, {"name": GRAPH_NAME}).consume()
except Exception:
    # 不存在就忽略
    pass

# -- Step 1: project 图投影（mp_Attribution 节点 + SIMILAR 关系）
PROJECT = """
CALL gds.graph.project(
  $name,
  'mp_Attribution',
  ['SIMILAR']
)
YIELD graphName, nodeCount, relationshipCount//, createMillis
RETURN graphName, nodeCount, relationshipCount//, createMillis
"""
with driver.session() as s:
    rec = s.run(PROJECT, {"name": GRAPH_NAME}).single()
    #print(f"[project] {rec['graphName']} nodes={rec['nodeCount']} rels={rec['relationshipCount']} in {rec['createMillis']}ms")
    print(f"[project] {rec['graphName']} nodes={rec['nodeCount']} rels={rec['relationshipCount']} ")
# -- Step 2: 在该投影上执行 WCC（演示返回前 20 行，便于观察）
WCC_STREAM_PREVIEW = """
CALL gds.wcc.stream($name)
YIELD nodeId, componentId
RETURN gds.util.asNode(nodeId).id AS nodeName, componentId
ORDER BY componentId, nodeName
LIMIT 20
"""
with driver.session() as s:
    print("[wcc preview] top 20 rows:")
    for r in s.run(WCC_STREAM_PREVIEW, {"name": GRAPH_NAME}):
        print(f"  component={r['componentId']}, nodeName={r['nodeName']}")

# -- Step 3: 基于 WCC 结果，为每个连通分量 MERGE 一个 master (pg:AttributionMaster) 并建立引用关系
#   逻辑与你给的 Cypher 保持一致：用 componentId 作 uid，首次创建时拷贝一个样本文本/名称属性
BUILD_MASTERS = """
CALL gds.wcc.stream($name)
YIELD nodeId, componentId
WITH gds.util.asNode(nodeId) AS mp_Attribution, componentId AS golden_id
MERGE (pg:AttributionMaster { uid: golden_id })
ON CREATE SET
  pg.originalText = mp_Attribution.WHU_HASORIGINALTEXT,
  pg.hasName     = mp_Attribution.WHU_HASNAME
WITH pg, mp_Attribution
MERGE (pg)-[:HAS_REFERENCE]->(mp_Attribution)
RETURN count(*) AS linked
"""
with driver.session() as s:
    rec = s.run(BUILD_MASTERS, {"name": GRAPH_NAME}).single()
    print(f"[masters] HAS_REFERENCE created/merged (rows touched): {rec['linked']}")

# -- Step 4a: 观测出度>MAX_OUT_DEG 的 master
CHECK_HIGH_OUTDEG = """
MATCH (startNode:AttributionMaster)-[r:HAS_REFERENCE]->(endNode)
WITH startNode, count(r) AS connectionCount
WHERE connectionCount > $maxDeg
RETURN elementId(startNode) AS masterId, connectionCount
ORDER BY connectionCount DESC
"""
with driver.session() as s:
    high = list(s.run(CHECK_HIGH_OUTDEG, {"maxDeg": MAX_OUT_DEG}))
    print(f"[check] masters with out-degree > {MAX_OUT_DEG}: {len(high)}")
    for r in high[:10]:
        print(f"  masterId={r['masterId']} outDegree={r['connectionCount']}")

# -- Step 4b: 删除出度>MAX_OUT_DEG 的 master（及其关系）
DELETE_HIGH_OUTDEG = """
MATCH (pg:AttributionMaster)
WHERE COUNT { (pg)-[:HAS_REFERENCE]->() } > $maxDeg
DETACH DELETE pg
"""
with driver.session() as s:
    summary = s.run(DELETE_HIGH_OUTDEG, {"maxDeg": MAX_OUT_DEG}).consume()
    # 统计删除数量（5.x 的统计在 counters 里）
    counters = summary.counters
    print(f"[cleanup] nodes deleted={counters.nodes_deleted}, rels deleted={counters.relationships_deleted}")

# （可选）用完可 drop 掉投影，释放内存
try:
    with driver.session() as s:
        s.run(DROP_GRAPH, {"name": GRAPH_NAME}).consume()
        print(f"[drop] dropped GDS graph: {GRAPH_NAME}")
except Exception:
    pass


[project] identity-wcc nodes=615 rels=1851 
[wcc preview] top 20 rows:
  component=0, nodeName=None
  component=1, nodeName=None
  component=1, nodeName=None
  component=2, nodeName=None
  component=2, nodeName=None
  component=3, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
  component=4, nodeName=None
[masters] HAS_REFERENCE created/merged (rows touched): 615
[check] masters with out-degree > 10: 4
  masterId=4:a588b8f8-4a1b-4a79-8a1a-924958fc2820:203 outDegree=154
  masterId=4:a588b8f8-4a1b-4a79-8a1a-924958fc2820:1686 outDegree=44
  masterId=4:a588b8f8-4a1b-4a79-8a1a-924958fc2820:245 outDegree=14
  masterId=4:a588b8f8-4a1b-4a79-8a1a-924958fc28

# pipeline code block C 节点 embdding，建立索引

## C11 embedding 融合准备，text embedding化，建立索引

neo4j 中的官方方法具有以下局限性：
   1. 其使用spacy作为语义融合的方法，但spacy维度过低，可以照成融合效果不好
   2. 没有使用链接，聚类等方法，特别是没有与本地知识库对齐

使用自定义的向量相似度进行融合
0. 为相关文本属性构建对应的embedding属性
1. 创建embedding 索引
2. 通过embedding 索引进行相似度计算

[包含了建立embedding索引的代码，通过cos 融合的文章](https://neo4j.com/blog/developer/property-graph-index-llamaindex/)


----

 

> 参考书籍：Building Knowledge Graphs A Practitioner’s Guide (Jesus Barrasa, Jim）

embedding 属性构建，index构建的基础上，实现多节点的融合

但这里的融合策略可以分为两种

1. 直接融合，即之前的相似节点全部去掉，直接融合为一个节点，但在新节点中保留之前融合节点的对应关系，属性，来源。
   
2. 不合并节点，而是通过  SAME_AS， SIMILAR 关系建立新的节点与融合简单的关系。这种方法也是neo4j推荐方法
 

这两种方法各有优缺点

* 第一种方法可以大大减少相关节点数目，是的kg更加简单，同时在图检索时，也使得需要跳的节点减少，检索效率更高，但缺点在于原始节点信息可能会丢失，同时可能无法溯源。

* 第二种方法的优点在于，原始节点的信息不会丢失，同时也可以溯源到原始节点，缺点在于，图中会存在大量的 sameas， SIMILAR 关系，使得图变得复杂。

但第二种方法与Agent结合度更高，通过外部的知识库（RDF）建立连接，实现更高精度的匹配

同时结合neo4j 方法，其使用了wcc 最弱关联算法得到相同的节点，可以作为 embedding的补充


以下首先实现的是基于embedding相似度的对齐方法，但提供 E1，E2 两种策略

* E1 直接融合
* E2 不合并节点，而是通过 SAME_AS， SIMILAR 关系建立新的节点与融合简单的关系。
  同时E2 的SAME_As 关系建立必须限制于特定的节点，包括
  * whu_Target_analyte 化学对象
  * whu_EnvironmentFeature 环境
  * whu_Reagen 化学品
  * whu_Device 设备
  * whu_Specimen 样品
  * envo_Material 物质
  * mp_Attribution 贡献者（作者，机构）
  * mp_References 引用
 而其他节点，例如mp_State,mp_Claim, 以及Expeirment类节点，理论上都只能建立SIMILAR 关系，而非SAME_AS 关系

 其愿意在于，可建立SAME_AS 节点，都是独立对象概念，而非文本描述，并且概念边界明晰，同时可以于外部的知识库建立，联系，从而实现更高的匹配精度。

 而mp_State,mp_Claim, 以及Expeirment，Activty节点，代表的是论证描述，或实验过程描述，本身具有较多的文本描述，概念边界也不宜确定，带有一定的不确定性。

----


实现折中的方法

1. 为了于本体看概念对齐，在此不使用SAME_AS 关系，而是使用skos:exactMatch / closeMatch 
>  本体语义上（比如 owl:sameAs），它是自反、对称、传递且语义极强的“同一性”声明；一旦断言错误，会带来级联污染（推理把不同事物当成一个）。因此实际工程里常用更温和的映射如 skos:exactMatch / closeMatch 表达“概念/词汇层的等同或近似”，而不是盲用 sameAs

2.mp_Attribution（作者/机构）和 mp_References（文献）只有在有权威 ID（ORCID、ROR、DOI/PMID/ISBN）时才应近似地视为“可 SAME_AS”；
这个问题的解决，只能在初始阶段，通过agent获取外部文献知识库（通过完整引用名称）来获取权威ID，然后再建立SAME_AS 关系，但难度相对较高，且目前对本研究影响不大，可以放置后期解决

3. 环境要素（whu_EnvironmentFeature）的对齐需要地理与层级规则

仅依赖文本/嵌入不够。应加入地理编码/行政区层级/空间缓冲等规则（例如经纬度、GeoHash、行政层级父子一致性），否则易把相邻地点或同名异地合并。

这个问题的解决，也需要在合并前，为地理节点建立丰富的地理属性，包括经纬度，geoid，层级关系，也需要通过agent实现。

4. Device,Regent 也是同样，必须满足厂商，型号完全相同，才能实现owl:sameAs 关系 ,否则不能建立

总结而言，实体对齐尽量采用E2 软方法，同时实现多种方法的结合，而非只使用embedding方式。并尽量构成pipeline

以下为neo4j 文献中提出的实体对齐step，可以作为参考：

 1. Create a SAME_AS relationship between nodes for which there is a match on a
strong identifier feature.

2. Create a weighted SIMILAR relationship between nodes for which there is a match
above threshold on a weak identifier feature and a SAME_AS relationship does not
exist between the two nodes.


3. Discard the SIMILAR relationships for which there is a mismatch in one of the
strong identifier features. The weight in the relationship will be the similarity
score.

4. Apply a correcting factor to the weight in a SIMILAR relationship when there is a
match above threshold on a nonidentifying feature.
5. Discard the SIMILAR relationships with a similarity score below a minimum
threshold.

结合以上步骤，本研究采取以下策略：

1. 为了于本体看概念对齐，在此不使用SAME_AS 关系，而是使用skos:exactMatch / closeMatch ,同时首先
   通过，whu_Target_analyte 化学对象，whu_Reagen 化学品，whu_Specimen 样品，基于agent 从外部知识库，后去iri，作为skos:exactMatch / closeMatch 的精确匹配属性

2. 对whu_EnvironmentFeature 环境要素，基于agent  从外部知识库，geo id，作为skos:exactMatch / closeMatch 的精确匹配属性，如果其为地理类别，地理环境则通过外部知识库获取iri进行作为skos:exactMatch / closeMatch 的精确匹配属性

4. 基于whu_Device 如果厂商，型号完全相同，才能实现owl:sameAs 关系 ,否则不能建立，
   同时基于whu_Reagen 如果名称完全相同，才能实现owl:sameAs 关系 ,否则不能建立

5. 其他依然基于embedding合并

***master entity***

合并的关键在于创建主实体（master entity），并将所有其他实体与主实体建立关系。即其为其他原实体所对齐的实体。

实际上，neo4j 的方法里面有很多问题，包括其得到的master entitiy属性是简单地通过第一个相关的节点得到，而不是通过算法得到，这还不如目前我通过 多个相同节点合并其原始内容得到的结果更有意义，同时由于其基于wcc算法，意味着但凡建立了联系节点，都可以认为是合并实体，与master entity 建立联系，这虽然与目前的做法相同，但是，其丧失了similar 中 score 语义，无法得到关联的强弱关系，但由于强弱关系是两两相对的，故也难以保持这一信息。这个问题可以之后考虑。

基于以上背景，以下是我设计的对齐方案，分为M1：概念类实体对齐；M2：描述类实体对齐

***概念类实体对齐***

M1 方案中由根据概念的类别，分为a:生化概念类实体，与b:地理，设备类，化学试剂两个类别

生化概念实体可以与现有的rdf建立紧密的skos:exactMatch / closeMatch 关系，同时可以直接基于名称进行对应

而b类别地理，设备，化学试剂，则由于存在属性，如名称，厂商，型号，等，并不能单纯的通过名称进行对应，其需要进一步的处理。


  * whu_Target_analyte 化学对象
  * whu_Specimen 样品
  * whu_ProcessedSpecimen 处理后样品

但两类的策略基本相同

1. agent读取a类节点，并根据名称，从外部知识库获取iri，作为skos:exactMatch / closeMatch 的精确匹配属性
2. 根据iri创建neo4j master entitiy，以及iri rdf中对应属性，作为外部知识扩充
3. 在neo4j中建立a类节点与 masert enitity 的skos:exactMatch 关系


使用embeding 进行融和在此不使用spacy的方法，而改用
1. 首先执行初步的single，fuzzy过滤，从而减少需要处理的node数目
2. 在kg中，需要融合的节点创建embedding属性，对其orgial text进行进行embeding
3. 计算不同节点的cos相似度，实现融合
 

 

实现图中WHU_HASNAME，`WHU_HASORIGINALTEXT 两个文本的embedding化

注意这里WHU_HASNAME是llm综合得到的内容名称，而WHU_HASORIGINALTEXT是原始文本

其流程如下：

``` mermaid
 
flowchart LR
    A([开始处理]) --> B[检查当前embedding状态]
    B --> C{有缺失embedding的节点?}
    C -- 否 --> D([所有节点已完成<br>流程结束])
    C -- 是 --> E[获取一批需处理的节点]
    E --> F[为每个节点创建文本]
    F --> G{有有效文本?}
    G -- 否 --> C
    G -- 是 --> H[批量生成embeddings]
    H --> I[更新数据库]
    I --> J[记录并报告本批处理结果]
    J --> K[检查是否全部完成]
    K -- 否 --> C
    K -- 是 --> L([所有节点处理完成!])
```


-----


同时由于原始node中，包含两个与text相关的文本属性，为此通过

``` python
    success = create_embeddings_for_properties(
        properties=["WHU_HASNAME", "WHU_HASORIGINALTEXT"],
        strategy="concatenate",
        batch_size=50,
        expected_dimension=768  # 指定期望维度，可选参数
    )
```
中strategy 参数控制实现

选择的 strategy 参数。OptimizedEmbeddingManager 类中的 create_text_for_embedding 方法提供了多种策略来选择或合成用于生成嵌入向量的文本。

``` mermaid
 flowchart TD
    A[调用create_text_for_embedding<br>传入node和strategy] --> B{判断strategy}
    
    B -- "concatenate" --> C[拼接策略]
    B -- "name_only" --> D[仅名称策略]
    B -- "text_only" --> E[仅文本策略]
    B -- "prefer_name" --> F[偏好名称策略]
    
    subgraph SG_Concatenate [拼接策略流程]
        direction TB
        C1[提取name与text]
        C2{"text ≠ name?"}
        C2 -- "是" --> C3[用' | '连接name和text]
        C2 -- "否" --> C4[仅保留name]
        C3 --> C_Output[返回合成文本]
        C4 --> C_Output
    end
    
    D --> D_Output[返回name]
    E --> E_Output[返回text]
    
    subgraph SG_PreferName [偏好名称策略流程]
        direction TB
        F1{"name存在?"}
        F1 -- "是" --> F_OutputName[返回name]
        F1 -- "否" --> F2{"text存在?"}
        F2 -- "是" --> F_OutputText[返回text]
        F2 -- "否" --> F_OutputNone[返回None]
    end
    
    C_Output --> G[得到用于生成<br>Embedding的最终文本]
    D_Output --> G
    E_Output --> G
    F_OutputName --> G
    F_OutputText --> G
    F_OutputNone --> G
 
```


从流程图中可以看出，strategy 参数是控制最终使用什么文本来生成 Embedding 的关键。以下是每种策略的详细说明：

策略 (strategy) 行为 适用场景

concatenate (默认) 将 name (WHU_HASNAME) 和 text (WHU_HASORIGINALTEXT) 用                      |  连接成一个字符串。若两者相同，则自动去重，只保留一个。 希望综合利用两个属性的信息，提供最丰富的上下文。

name_only 只使用 name (WHU_HASNAME) 属性来生成embedding。text 属性被忽略。 确信名称属性已包含足够信息，或文本属性噪音较大时。

text_only 只使用 text (WHU_HASORIGINALTEXT) 属性来生成embedding。name 属性被忽略。 确信文本属性包含更详细、更准确的信息时。

prefer_name 优先使用 name (WHU_HASNAME)。只有当 name 为空时，才回退使用 text (WHU_HASORIGINALTEXT)。 名称属性是主要标识，文本属性作为补充后备信息时。

在代码最后的 main() 函数示例中，明确使用了 strategy="concatenate"，这意味着在这个默认的示例场景中，embedding 是基于 WHU_HASNAME 和 WHU_HASORIGINALTEXT 两个属性合并后的文本 生成的。



•   如何控制：最终 embedding 的生成来源并非固定不变，完全由 create_text_for_embedding 方法的 strategy 参数决定。

•   默认行为：如示例中所用，默认策略 (concatenate) 会合并两个属性的文本，以期获得信息量最大、上下文最全的表示。

•   灵活性与选择：可根据数据质量和具体任务的需求，通过更改 strategy 参数来选择最合适的文本生成策略。例如，如果 WHU_HASORIGINALTEXT 字段质量很高且包含主要信息，可以选用 text_only；若 WHU_HASNAME 是唯一可靠的标识，则可选用 name_only 或 prefer_name。


**代码块功能**

实现对__Entity__和__Community__中

In [4]:
import utilities.return_llm_database
import importlib
import torch
importlib.reload(utilities.return_llm_database)
class OptimizedEmbeddingManager:
    """优化的Embedding管理器 - 支持指定属性"""
    
    def __init__(self, expected_dimension=None):
        """初始化管理器"""
        print("连接数据库和模型...")
        manager = utilities.return_llm_database.DatabaseManager()
        self.llm, self.embed_model, self.neo4j_driver = manager.get_components()
        
        # 设置期望的embedding维度
        self.expected_dimension = expected_dimension
        
        # 检查GPU环境
        self._check_gpu_environment()
        
        # 检测模型的实际维度
        self._detect_embedding_dimension()
        print("初始化完成")
    
    def _detect_embedding_dimension(self):
        """检测embedding模型的实际维度"""
        try:
            # 适配SentenceTransformerEmbeddings的API
            if hasattr(self.embed_model, 'embed_query'):
                test_embedding = self.embed_model.embed_query("测试")
            elif hasattr(self.embed_model, 'embed_documents'):
                test_embedding = self.embed_model.embed_documents(["测试"])[0]
            else:
                # 尝试直接调用
                test_embedding = self.embed_model("测试")
            
            if hasattr(test_embedding, 'tolist'):
                test_embedding = test_embedding.tolist()
            elif torch.is_tensor(test_embedding):
                test_embedding = test_embedding.cpu().tolist() if test_embedding.is_cuda else test_embedding.tolist()
            
            actual_dim = len(test_embedding)
            print(f"模型实际维度: {actual_dim}")
            
            if self.expected_dimension is None:
                self.expected_dimension = actual_dim
                print(f"设置期望维度为: {self.expected_dimension}")
            elif self.expected_dimension != actual_dim:
                print(f"警告: 期望维度({self.expected_dimension}) != 实际维度({actual_dim})")
                self.expected_dimension = actual_dim
                
        except Exception as e:
            print(f"检测维度失败: {e}")
            if self.expected_dimension is None:
                self.expected_dimension = 768  # 默认维度
    
    def _check_gpu_environment(self):
        """检查GPU环境"""
        if torch.cuda.is_available():
            self.device = 'cuda'
            print(f"GPU可用: {torch.cuda.get_device_name(0)}")
            torch.cuda.empty_cache()
        else:
            self.device = 'cpu'
            print("使用CPU")
    
    def get_nodes_for_embedding(self, properties=None, limit=100):
        """获取需要创建embedding的节点"""
        if properties is None:
            properties = ["WHU_HASNAME", "WHU_HASORIGINALTEXT"]
        
        # 构建WHERE条件 - 至少一个属性有值或者有summary但没有embedding
        property_conditions = []
        for prop in properties:
            property_conditions.append(f"n.{prop} IS NOT NULL AND trim(toString(n.{prop})) <> ''")
        
        # 添加summary条件（注意：不使用反引号）
        property_conditions.append("n.summary IS NOT NULL AND trim(toString(n.summary)) <> ''")
        
        # 跳过Chunk节点和已有embedding的节点
        where_clause = f"({' OR '.join(property_conditions)}) AND n.embedding IS NULL AND NOT 'Chunk' IN labels(n)"
        
        query = f"""
        MATCH (n)
        WHERE (n:__Entity__ OR n:__Community__) AND {where_clause}
        RETURN elementId(n) as id, 
               n.WHU_HASNAME as name,
               n.WHU_HASORIGINALTEXT as text,
               n.summary as summary,
               n.title as title,
               labels(n) as labels
        LIMIT $limit
        """
        
        print(f"\n=== 调试信息 ===")
        print(f"执行查询: {query}")
        print(f"WHERE条件: {where_clause}")
        
        with self.neo4j_driver.session() as session:
            try:
                result = session.run(query, limit=limit)
                records = [record for record in result]
                print(f"找到 {len(records)} 个待处理节点")
                
                # 打印前3个节点的详细信息
                for i, record in enumerate(records[:3]):
                    print(f"\n节点 {i+1}:")
                    print(f"  - ID: {record['id']}")
                    print(f"  - Labels: {record.get('labels', [])}")
                    print(f"  - Name: {record.get('name', 'None')}")
                    print(f"  - Text: {str(record.get('text', 'None'))[:50]}...")
                    print(f"  - Summary: {str(record.get('summary', 'None'))[:50]}...")
                    print(f"  - Title: {record.get('title', 'None')}")
                
                return records
            except Exception as e:
                print(f"获取节点失败: {e}")
                import traceback
                traceback.print_exc()
                return []
    
    def create_text_for_embedding(self, node, strategy="concatenate"):
        """为节点创建用于embedding的文本"""
        name = node.get('name') or ""
        text = node.get('text') or ""
        summary = node.get('summary') or ""
        title = node.get('title') or ""
        
        # 清理文本
        name = str(name).strip() if name else ""
        text = str(text).strip() if text else ""
        summary = str(summary).strip() if summary else ""
        title = str(title).strip() if title else ""
        
        print(f"\n[DEBUG] 创建embedding文本:")
        print(f"  - name: '{name[:50] if name else 'None'}...'")
        print(f"  - text: '{text[:50] if text else 'None'}...'")
        print(f"  - title: '{title[:50] if title else 'None'}...'")
        print(f"  - summary: '{summary[:50] if summary else 'None'}...'")
        
        if strategy == "concatenate":
            # 拼接属性和summary
            parts = []
            if name:
                parts.append(name)
            if text and text != name:  # 避免重复
                parts.append(text)
            if title and title not in [name, text]:  # 添加title
                parts.append(title)
            if summary and summary not in [name, text, title]:  # 避免重复
                parts.append(summary)
            
            result = " | ".join(parts) if parts else None
            print(f"  - 最终文本: '{result[:100] if result else 'None'}...'")
            return result
            
        elif strategy == "name_only":
            return name if name else None
            
        elif strategy == "text_only":
            return text if text else None
            
        elif strategy == "prefer_name":
            return name if name else (text if text else None)
            
        else:
            raise ValueError(f"未知策略: {strategy}")
    
    def generate_embeddings_batch(self, texts):
        """批量生成embeddings"""
        if not texts:
            return []
        
        try:
            # 适配SentenceTransformerEmbeddings的API
            if hasattr(self.embed_model, 'embed_documents'):
                # LangChain的embed_documents方法
                embeddings = self.embed_model.embed_documents(texts)
            elif hasattr(self.embed_model, 'embed_query'):
                # 逐个调用embed_query
                embeddings = [self.embed_model.embed_query(text) for text in texts]
            else:
                # 直接调用
                embeddings = [self.embed_model(text) for text in texts]
            
            # 确保返回list格式
            processed_embeddings = []
            for emb in embeddings:
                if hasattr(emb, 'tolist'):
                    processed_embeddings.append(emb.tolist())
                elif torch.is_tensor(emb):
                    processed_embeddings.append(emb.cpu().tolist() if emb.is_cuda else emb.tolist())
                else:
                    processed_embeddings.append(emb)
            
            print(f"成功生成 {len(processed_embeddings)} 个embedding")
            return processed_embeddings
            
        except Exception as e:
            print(f"批量生成失败: {e}")
            # 回退到逐个生成
            return self._generate_embeddings_one_by_one(texts)
    
    def _generate_embeddings_one_by_one(self, texts):
        """逐个生成embedding"""
        print("回退到逐个处理...")
        embeddings = []
        
        for i, text in enumerate(texts):
            try:
                # 适配SentenceTransformerEmbeddings的API
                if hasattr(self.embed_model, 'embed_query'):
                    embedding = self.embed_model.embed_query(text)
                elif hasattr(self.embed_model, 'embed_documents'):
                    embedding = self.embed_model.embed_documents([text])[0]
                else:
                    embedding = self.embed_model(text)
                
                # 转换为list
                if hasattr(embedding, 'tolist'):
                    embedding = embedding.tolist()
                elif torch.is_tensor(embedding):
                    embedding = embedding.cpu().tolist() if embedding.is_cuda else embedding.tolist()
                
                embeddings.append(embedding)
                
                if (i + 1) % 10 == 0:
                    print(f"已处理 {i + 1}/{len(texts)}")
                    
            except Exception as e:
                print(f"处理失败: {text[:50]}... 错误: {e}")
                continue
        
        return embeddings
    
    def update_node_embeddings(self, node_ids, embeddings):
        """批量更新节点embeddings，验证维度"""
        if len(node_ids) != len(embeddings):
            print(f"长度不匹配: {len(node_ids)} vs {len(embeddings)}")
            return 0
        
        success_count = 0
        
        with self.neo4j_driver.session() as session:
            for node_id, embedding in zip(node_ids, embeddings):
                try:
                    # 验证embedding维度
                    if len(embedding) != self.expected_dimension:
                        print(f"维度错误: 期望{self.expected_dimension}, 实际{len(embedding)}, 跳过节点 {node_id}")
                        continue
                    
                    session.run("""
                    MATCH (n) WHERE elementId(n) = $node_id
                    SET n.embedding = $embedding
                    """, node_id=node_id, embedding=embedding)
                    success_count += 1
                except Exception as e:
                    print(f"更新节点 {node_id} 失败: {e}")
        
        return success_count
    
    def check_embedding_status(self, properties=None):
        """检查embedding状态"""
        if properties is None:
            properties = ["WHU_HASNAME", "WHU_HASORIGINALTEXT"]
        
        # 构建WHERE条件
        property_conditions = []
        for prop in properties:
            property_conditions.append(f"n.{prop} IS NOT NULL AND trim(toString(n.{prop})) <> ''")
        
        # 添加summary条件
        property_conditions.append("n.summary IS NOT NULL AND trim(toString(n.summary)) <> ''")
        
        where_clause = ' OR '.join(property_conditions)
        
        with self.neo4j_driver.session() as session:
            try:
                # 总节点数（排除Chunk节点，包含__Entity__和__Community__）
                total_result = session.run(f"""
                MATCH (n)
                WHERE (n:__Entity__ OR n:__Community__) AND ({where_clause}) AND NOT 'Chunk' IN labels(n)
                RETURN count(n) as total
                """)
                total = total_result.single()["total"]
                
                # 有embedding的节点数（排除Chunk节点，包含__Entity__和__Community__）
                has_embedding_result = session.run(f"""
                MATCH (n)
                WHERE (n:__Entity__ OR n:__Community__) AND ({where_clause}) AND NOT 'Chunk' IN labels(n) AND n.embedding IS NOT NULL
                RETURN count(n) as has_embedding
                """)
                has_embedding = has_embedding_result.single()["has_embedding"]
                
                missing = total - has_embedding
                rate = (has_embedding / total * 100) if total > 0 else 0
                
                print(f"状态: 总节点{total}, 有embedding{has_embedding}, 缺失{missing}, 完成率{rate:.1f}%")
                return {'total': total, 'has_embedding': has_embedding, 'missing': missing, 'rate': rate}
                
            except Exception as e:
                print(f"检查状态失败: {e}")
                return {'total': 0, 'has_embedding': 0, 'missing': 0, 'rate': 0}
    
    def process_embeddings(self, properties=None, strategy="concatenate", batch_size=50):
        """处理embeddings的主函数"""
        if properties is None:
            properties = ["WHU_HASNAME", "WHU_HASORIGINALTEXT"]
        
        print(f"开始处理embeddings - 属性: {properties}, 策略: {strategy}")
        
        # 检查初始状态
        status = self.check_embedding_status(properties)
        if status['missing'] == 0:
            print("所有节点已有embedding")
            return True
        
        total_processed = 0
        
        while True:
            # 获取需要处理的节点
            nodes = self.get_nodes_for_embedding(properties, batch_size)
            if not nodes:
                print("没有更多节点需要处理")
                break
            
            print(f"处理 {len(nodes)} 个节点...")
            
            # 创建文本
            texts = []
            valid_nodes = []
            
            for node in nodes:
                text = self.create_text_for_embedding(node, strategy)
                if text:
                    texts.append(text)
                    valid_nodes.append(node)
            
            if not texts:
                print("没有有效文本")
                break
            
            # 清理GPU缓存
            if self.device == 'cuda':
                torch.cuda.empty_cache()
            
            # 生成embeddings
            embeddings = self.generate_embeddings_batch(texts)
            
            if not embeddings:
                print("生成embedding失败")
                break
            
            # 更新数据库
            node_ids = [node['id'] for node in valid_nodes[:len(embeddings)]]
            updated = self.update_node_embeddings(node_ids, embeddings)
            
            total_processed += updated
            print(f"本批成功处理: {updated}/{len(valid_nodes)}")
            
            # 检查是否完成
            current_status = self.check_embedding_status(properties)
            if current_status['missing'] == 0:
                print("所有节点处理完成!")
                break
        
        print(f"处理完成，总计: {total_processed} 个节点")
        return total_processed > 0

def create_embeddings_for_properties(properties=None, strategy="concatenate", batch_size=50, expected_dimension=None):
    """简化的调用函数"""
    manager = OptimizedEmbeddingManager(expected_dimension)
    return manager.process_embeddings(properties, strategy, batch_size)

# 使用示例
def main():
    """主函数"""
    
    # 方案1：拼接两个属性和summary（如果有），指定维度
    print("=== 方案1：拼接WHU_HASNAME、WHU_HASORIGINALTEXT和summary ===")
    success = create_embeddings_for_properties(
        properties=["WHU_HASNAME", "WHU_HASORIGINALTEXT"],
        strategy="concatenate",
        batch_size=50,
        expected_dimension=768  # 指定期望维度，可选参数
    )
    
    if success:
        print("embedding创建成功")
    else:
        print("embedding创建失败或无需处理")

if __name__ == "__main__":
    main()

=== 方案1：拼接WHU_HASNAME、WHU_HASORIGINALTEXT和summary ===
连接数据库和模型...
使用CPU
模型实际维度: 768
初始化完成
开始处理embeddings - 属性: ['WHU_HASNAME', 'WHU_HASORIGINALTEXT'], 策略: concatenate
状态: 总节点20191, 有embedding0, 缺失20191, 完成率0.0%

=== 调试信息 ===
执行查询: 
        MATCH (n)
        WHERE (n:__Entity__ OR n:__Community__) AND (n.WHU_HASNAME IS NOT NULL AND trim(toString(n.WHU_HASNAME)) <> '' OR n.WHU_HASORIGINALTEXT IS NOT NULL AND trim(toString(n.WHU_HASORIGINALTEXT)) <> '' OR n.summary IS NOT NULL AND trim(toString(n.summary)) <> '') AND n.embedding IS NULL AND NOT 'Chunk' IN labels(n)
        RETURN elementId(n) as id, 
               n.WHU_HASNAME as name,
               n.WHU_HASORIGINALTEXT as text,
               n.summary as summary,
               n.title as title,
               labels(n) as labels
        LIMIT $limit
        
WHERE条件: (n.WHU_HASNAME IS NOT NULL AND trim(toString(n.WHU_HASNAME)) <> '' OR n.WHU_HASORIGINALTEXT IS NOT NULL AND trim(toString(n.WHU_HASORIGINALTEXT)) <> '' OR n.summary IS

##C11.1 实现relation 的embdding，方式与node相同

In [5]:
import utilities.return_llm_database
import importlib
import torch
importlib.reload(utilities.return_llm_database)

class RelationEmbeddingManager:
    """关系Embedding管理器"""
    
    def __init__(self, expected_dimension=768):
        """初始化管理器"""
        print("连接数据库和模型...")
        manager = utilities.return_llm_database.DatabaseManager()
        self.llm, self.embed_model, self.neo4j_driver = manager.get_components()
        self.expected_dimension = expected_dimension
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"使用设备: {self.device}")
        print("初始化完成")
    
    def get_relations_for_embedding(self, limit=100):
        """获取需要创建embedding的关系"""
        query = """
        MATCH (s)-[r]->(t)
        WHERE (r.WHU_HASNAME IS NOT NULL AND trim(toString(r.WHU_HASNAME)) <> ''
               OR r.WHU_HASORIGINALTEXT IS NOT NULL AND trim(toString(r.WHU_HASORIGINALTEXT)) <> '')
              AND r.embedding IS NULL
        RETURN elementId(r) as id,
               r.WHU_HASNAME as name,
               r.WHU_HASORIGINALTEXT as text,
               type(r) as rel_type
        LIMIT $limit
        """
        
        with self.neo4j_driver.session() as session:
            result = session.run(query, limit=limit)
            records = list(result)
            print(f"找到 {len(records)} 个待处理关系")
            
            # 显示前3个关系的详细信息
            for i, record in enumerate(records[:3]):
                print(f"  关系 {i+1}: {record['rel_type']}")
                print(f"    - Name: {str(record.get('name', 'None'))[:50]}")
                print(f"    - Text: {str(record.get('text', 'None'))[:50]}")
            
            return records
    
    def create_text_for_embedding(self, relation):
        """为关系创建用于embedding的文本"""
        name = relation.get('name') or ""
        text = relation.get('text') or ""
        
        # 清理文本
        name = str(name).strip() if name else ""
        text = str(text).strip() if text else ""
        
        # 拼接策略：name | text
        parts = []
        if name:
            parts.append(name)
        if text and text != name:  # 避免重复
            parts.append(text)
        
        return " | ".join(parts) if parts else None
    
    def generate_embeddings_batch(self, texts):
        """批量生成embeddings"""
        if not texts:
            return []
        
        try:
            embeddings = self.embed_model.embed_documents(texts)
            
            # 转换为list格式
            processed = []
            for emb in embeddings:
                if hasattr(emb, 'tolist'):
                    processed.append(emb.tolist())
                elif torch.is_tensor(emb):
                    processed.append(emb.cpu().tolist())
                else:
                    processed.append(emb)
            
            print(f"成功生成 {len(processed)} 个embedding")
            return processed
            
        except Exception as e:
            print(f"批量生成失败: {e}，回退到逐个处理")
            return self._generate_one_by_one(texts)
    
    def _generate_one_by_one(self, texts):
        """逐个生成embedding（回退方案）"""
        embeddings = []
        for i, text in enumerate(texts):
            try:
                embedding = self.embed_model.embed_query(text)
                
                if hasattr(embedding, 'tolist'):
                    embedding = embedding.tolist()
                elif torch.is_tensor(embedding):
                    embedding = embedding.cpu().tolist()
                
                embeddings.append(embedding)
                
                if (i + 1) % 10 == 0:
                    print(f"已处理 {i + 1}/{len(texts)}")
                    
            except Exception as e:
                print(f"处理失败: {text[:50]}... 错误: {e}")
        
        return embeddings
    
    def update_relation_embeddings(self, rel_ids, embeddings):
        """批量更新关系embeddings"""
        if len(rel_ids) != len(embeddings):
            print(f"长度不匹配: {len(rel_ids)} vs {len(embeddings)}")
            return 0
        
        success_count = 0
        with self.neo4j_driver.session() as session:
            for rel_id, embedding in zip(rel_ids, embeddings):
                try:
                    # 验证维度
                    if len(embedding) != self.expected_dimension:
                        print(f"维度错误: 期望{self.expected_dimension}, 实际{len(embedding)}")
                        continue
                    
                    session.run("""
                    MATCH ()-[r]->() WHERE elementId(r) = $rel_id
                    SET r.embedding = $embedding
                    """, rel_id=rel_id, embedding=embedding)
                    
                    success_count += 1
                    
                except Exception as e:
                    print(f"更新关系 {rel_id} 失败: {e}")
        
        return success_count
    
    def check_status(self):
        """检查关系embedding状态"""
        with self.neo4j_driver.session() as session:
            # 总关系数
            total = session.run("""
            MATCH ()-[r]->()
            WHERE r.WHU_HASNAME IS NOT NULL AND trim(toString(r.WHU_HASNAME)) <> ''
                  OR r.WHU_HASORIGINALTEXT IS NOT NULL AND trim(toString(r.WHU_HASORIGINALTEXT)) <> ''
            RETURN count(r) as total
            """).single()["total"]
            
            # 有embedding的关系数
            has_embedding = session.run("""
            MATCH ()-[r]->()
            WHERE (r.WHU_HASNAME IS NOT NULL AND trim(toString(r.WHU_HASNAME)) <> ''
                   OR r.WHU_HASORIGINALTEXT IS NOT NULL AND trim(toString(r.WHU_HASORIGINALTEXT)) <> '')
                  AND r.embedding IS NOT NULL
            RETURN count(r) as count
            """).single()["count"]
            
            missing = total - has_embedding
            rate = (has_embedding / total * 100) if total > 0 else 0
            
            print(f"关系状态: 总数{total}, 有embedding{has_embedding}, 缺失{missing}, 完成率{rate:.1f}%")
            return {'total': total, 'has_embedding': has_embedding, 'missing': missing, 'rate': rate}
    
    def process(self, batch_size=50):
        """处理关系embeddings的主函数"""
        print("\n=== 开始处理关系embedding ===")
        
        # 检查初始状态
        status = self.check_status()
        if status['missing'] == 0:
            print("所有关系已有embedding")
            return True
        
        total_processed = 0
        
        while True:
            # 获取待处理关系
            relations = self.get_relations_for_embedding(batch_size)
            
            if not relations:
                print("没有更多关系需要处理")
                break
            
            # 创建文本
            texts = []
            valid_relations = []
            
            for relation in relations:
                text = self.create_text_for_embedding(relation)
                if text:
                    texts.append(text)
                    valid_relations.append(relation)
                else:
                    print(f"跳过关系 {relation['id']}：无有效文本")
            
            if not texts:
                print("没有有效文本")
                break
            
            print(f"处理 {len(texts)} 个关系...")
            
            # 清理GPU缓存
            if self.device == 'cuda':
                torch.cuda.empty_cache()
            
            # 生成embeddings
            embeddings = self.generate_embeddings_batch(texts)
            
            if not embeddings:
                print("生成embedding失败")
                break
            
            # 更新数据库
            rel_ids = [rel['id'] for rel in valid_relations[:len(embeddings)]]
            updated = self.update_relation_embeddings(rel_ids, embeddings)
            
            total_processed += updated
            print(f"本批成功: {updated}/{len(valid_relations)}")
            
            # 检查是否完成
            current_status = self.check_status()
            if current_status['missing'] == 0:
                print("所有关系处理完成!")
                break
        
        print(f"\n处理完成，总计: {total_processed} 个关系")
        return total_processed > 0
    
    def close(self):
        """关闭连接"""
        if self.neo4j_driver:
            self.neo4j_driver.close()

# 简化的调用函数
def create_relation_embeddings(batch_size=50, expected_dimension=768):
    """创建关系embeddings"""
    manager = RelationEmbeddingManager(expected_dimension)
    success = manager.process(batch_size)
    manager.close()
    return success

# 主函数
if __name__ == "__main__":
    print("=== 开始处理关系embedding ===")
    success = create_relation_embeddings(batch_size=50)
    
    if success:
        print("\n✅ 关系embedding处理完成")
    else:
        print("\n⚠️ 关系embedding处理失败或无需处理")

=== 开始处理关系embedding ===
连接数据库和模型...
使用设备: cpu
初始化完成

=== 开始处理关系embedding ===
关系状态: 总数8406, 有embedding0, 缺失8406, 完成率0.0%
找到 50 个待处理关系
  关系 1: p_plan_isOutputVarOf
    - Name: sulfur-zinc co-modification output
    - Text: the co-modification of sulfur and zinc significant
  关系 2: p_plan_isOutputVarOf
    - Name: material preparation output
    - Text: mercury adsorption by BC, SBC, ZBC, and SZ-BC are 
  关系 3: prov_wasDerivedFrom
    - Name: precursor from corn stover
    - Text: the corn stover powder was mixed with ZnCl2 soluti
处理 50 个关系...
批量生成失败: 'SentenceTransformerEmbeddings' object has no attribute 'embed_documents'，回退到逐个处理
已处理 10/50
已处理 20/50
已处理 30/50
已处理 40/50
已处理 50/50
本批成功: 50/50
关系状态: 总数8406, 有embedding50, 缺失8356, 完成率0.6%
找到 50 个待处理关系
  关系 1: dcterms_hasPart
    - Name: mercury concentration inclusion
    - Text: 200 mL solution containing 10 mg/L of HgCl2. The c
  关系 2: dcterms_hasPart
    - Name: solution volume inclusion
    - Text: 200 mL solution containing 10 mg/L of H

##  C12 建立索引
 

建立entity_embedding_index 索引，要目的是为了大幅提升基于节点嵌入向量（embedding）的相似性搜索效率，从而支持快速的知识检索和查询

**__Entitiy__ 代指所有的节点**
### C12.1 建立节点索引
在此关系暂时不建立embdding索引，原因在于，neo4j可能不支持，同时意义并不是很大

In [1]:
# 安装 Transformers 库
#! pip install transformers -i https://pypi.tuna.tsinghua.edu.cn/simple

# 安装 Sentence Transformers 嵌入模型库
#! pip install sentence-transformers -i https://pypi.tuna.tsinghua.edu.cn/simple
# 安装 LlamaIndex 核心包及 OpenAI、HuggingFace 组件
! pip install llama-index-core llama-index-llms-openai llama-index-embeddings-openai llama-index-embeddings-huggingface -i https://pypi.tuna.tsinghua.edu.cn/simple

# 安装 LlamaIndex 的 OpenAI 集成 (旧版风格)
! pip install llama_index.embeddings.openai -i https://pypi.tuna.tsinghua.edu.cn/simple
! pip install llama_index.llms.openai -i https://pypi.tuna.tsinghua.edu.cn/simple
! pip install llama_index.embeddings.huggingface -i https://pypi.tuna.tsinghua.edu.cn/simple
# 安装 LangChain 的 OpenAI 集成包
! pip install langchain-openai -i https://pypi.tuna.tsinghua.edu.cn/simple

# 安装 Neo4j 数据库驱动和 GraphRAG 组件
! pip install neo4j -i https://pypi.tuna.tsinghua.edu.cn/simple
! pip install neo4j_graphrag.llm -i https://pypi.tuna.tsinghua.edu.cn/simple
! pip install neo4j_graphrag.embeddings.sentence_transformers -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


ERROR: Could not find a version that satisfies the requirement neo4j_graphrag.llm (from versions: none)
ERROR: No matching distribution found for neo4j_graphrag.llm


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


ERROR: Could not find a version that satisfies the requirement neo4j_graphrag.embeddings.sentence_transformers (from versions: none)
ERROR: No matching distribution found for neo4j_graphrag.embeddings.sentence_transformers


In [2]:

## 构建index##
from neo4j import GraphDatabase
from utilities import return_llm_database as r


def create_vector_index(driver, name, label, embedding_property, dimensions, similarity_fn="cosine"):
    """创建向量索引，如果存在同名索引则先删除"""
    with driver.session() as session:
        try:
            # 检查索引是否存在
            check_query = "SHOW INDEXES YIELD name WHERE name = $name RETURN name"
            result = session.run(check_query, name=name)
            
            if result.single():
                print(f"发现已存在的索引: {name}，正在删除...")
                session.run(f"DROP INDEX {name} IF EXISTS")
                print(f"索引 {name} 已删除")
            
            # 创建新索引
            print(f"正在创建索引: {name}...")
            create_query = f"""
            CREATE VECTOR INDEX {name} IF NOT EXISTS
            FOR (n:{label})
            ON n.{embedding_property}
            OPTIONS {{
                indexConfig: {{
                    `vector.dimensions`: {dimensions},
                    `vector.similarity_function`: '{similarity_fn}'
                }}
            }}
            """
            session.run(create_query)
            print(f"索引 {name} 创建成功")
            
        except Exception as e:
            print(f"索引操作失败: {e}")
            raise


# 主执行流程
if __name__ == "__main__":
    # 获取 Neo4j 驱动（本地数据库）
    neo4j_driver = r.DatabaseManager.get_neo4j_driver(remotedatebase=False)
    
    # 创建向量索引
    create_vector_index(
        driver=neo4j_driver,
        name="entity_embedding_index",
        label="__Entity__",
        embedding_property="embedding",
        dimensions=768,  # 注意维度必须与构建时保持一致
        similarity_fn="cosine"
    )
    
    print("\n==== 创建索引完成 ====")
 

KeyboardInterrupt: 

## C13 相似节点查询

进行embeding相似度查询但不融合，列出融合候选

1. 以下在建立的embedding index 基础上实现相似度查询
2. 其查询的核心是cos，但也融合了字符查询，简化版本中可以去掉字符查询，完全依靠向量相似度，但存在的可能是过于依赖语义，得到的相似节点过多的情况
```cypher
      AND (toLower(node.name) CONTAINS toLower(e.name) OR toLower(e.name) CONTAINS toLower(node.name)
           OR apoc.text.distance(toLower(node.name), toLower(e.name)) < $distance)+
```

 返回结果只有whu_hasname等属性，没有id


In [3]:
 
 
import pprint as pp

 
import utilities.return_llm_database as db_manager
import pprint as pp

#manager = utilities.return_llm_database.DatabaseManager()
#llm, embed_model, neo4j_driver = manager.get_components()
neo4j_driver=db_manager.DatabaseManager.get_neo4j_driver(remotedatebase=False)

similarity_threshold = 0.90
word_edit_distance = 5  # 目前没用到，保留参数无所谓

CYPHER = """
// 主查询：寻找并聚合具有相似嵌入向量的实体节点
MATCH (e:__Entity__)
WHERE e.embedding IS NOT NULL
  AND coalesce(e.WHU_HASNAME, e.WHU_HASORIGINALTEXT, '') <> ''   // e 必须要有可用名称

// 使用子查询为每个实体e寻找前10个最相似的候选节点
CALL {
  WITH e
  // 注意：索引名与上面的 CREATE 一致：'entity_embedding_index'
  // 使用向量索引进行近似最近邻 (ANN) 搜索，相似度分数需高于阈值
  CALL db.index.vector.queryNodes('entity_embedding_index', 10, e.embedding)
  YIELD node, score
  WITH e, node, score
  WHERE node.embedding IS NOT NULL
    AND score > toFloat($cutoff)                 // 筛选高于相似度阈值的节点
    AND labels(e) = labels(node)                  // 确保候选节点与源节点标签相同
    AND coalesce(node.WHU_HASNAME, node.WHU_HASORIGINALTEXT, '') <> ''   // 候选也必须有可用名称
  WITH node, score
  ORDER BY coalesce(node.WHU_HASNAME, node.WHU_HASORIGINALTEXT, '') // 按名称排序以便后续处理
  RETURN collect(node) AS nodes                   // 将候选节点收集到列表中
}

// 主查询继续
WITH DISTINCT nodes                               // 确保候选节点列表唯一
WHERE size(nodes) > 1                            // 只处理找到多于一个候选节点的簇

// 将每个候选簇中的节点转换为它们的名称（使用coalesce确保非空），形成一个名称列表
WITH collect([n IN nodes | coalesce(n.WHU_HASNAME, n.WHU_HASORIGINALTEXT)]) AS results

// 展开结果列表的索引，以便后续两两比较
UNWIND range(0, size(results)-1, 1) AS index
WITH results, index, results[index] AS result
// 核心操作：合并具有交集的集合（通过apoc.coll.intersection检测）
// 使用reduce函数迭代合并所有与当前result有交集的其他results[index2]
WITH apoc.coll.sort(                             // 对最终合并后的集合进行排序
  reduce(
    acc = result,                                // 初始累加器为当前result
    index2 IN range(0, size(results)-1, 1) |     // 遍历所有其他集合
      CASE
        // 如果当前索引不等于遍历索引且两个集合有交集，则合并（取并集）
        WHEN index <> index2 AND size(apoc.coll.intersection(acc, results[index2])) > 0
          THEN apoc.coll.union(acc, results[index2])
        ELSE acc                                 // 否则保持累加器不变
      END
  )
) AS combinedResult

// 对合并后的结果进行去重
WITH DISTINCT combinedResult

// 进一步去重：过滤掉那些完全被其他更大集合包含的集合（避免冗余）
WITH collect(combinedResult) AS allCombinedResults
UNWIND range(0, size(allCombinedResults)-1, 1) AS combinedResultIndex
WITH allCombinedResults[combinedResultIndex] AS combinedResult, combinedResultIndex, allCombinedResults
WHERE NOT any(x IN range(0, size(allCombinedResults)-1, 1) // 检查是否存在另一个集合x
  WHERE x <> combinedResultIndex                          // x不是当前集合
    AND apoc.coll.containsAll(allCombinedResults[x], combinedResult) // 且x完全包含当前集合
)
// 返回最终去重后的合并结果（每个结果是一个相似实体的名称列表）
RETURN combinedResult
"""

records, summary, keys = neo4j_driver.execute_query(
    CYPHER,
    {"cutoff": similarity_threshold, "distance": word_edit_distance},
)

#for rec in records:
#    pp.pprint(rec["combinedResult"])
#    pp.pprint("#################THE PAIR OF SIMILARITY NODES###################")

 
# === 新增：将结果整理为 DataFrame 友好展示 ===
import pandas as pd
from IPython.display import display

# 提取簇结果（每条记录的 combinedResult 是一个“相似实体名称列表”）
clusters = [rec["combinedResult"] for rec in records]

# 规整每个簇：去空值、去重、排序，生成多列便于浏览与导出
rows = []
for i, cluster in enumerate(clusters, start=1):
    members = sorted({m for m in cluster if m})        # 去重 + 去空
    rows.append({
        "cluster_id": i,
        "size": len(members),
        "members": members,                            # 列表形式，便于后续处理
        "members_str": " | ".join(members),            # 便于快速浏览/导出
    })

df = pd.DataFrame(rows).sort_values(
    by=["size", "cluster_id"], ascending=[False, True]
).reset_index(drop=True)

# 在 Jupyter 中友好显示
display(df)

# 如需导出，可启用下面两行：
# out_path = "similar_entity_clusters.csv"
# df.to_csv(out_path, index=False, encoding="utf-8-sig")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL (e) { ... }} {position: line: 8, column: 1, offset: 187} for query: "\n// 主查询：寻找并聚合具有相似嵌入向量的实体节点\nMATCH (e:__Entity__)\nWHERE e.embedding IS NOT NULL\n  AND coalesce(e.WHU_HASNAME, e.WHU_HASORIGINALTEXT, '') <> ''   // e 必须要有可用名称\n\n// 使用子查询为每个实体e寻找前10个最相似的候选节点\nCALL {\n  WITH e\n  // 注意：索引名与上面的 CREATE 一致：'entity_embedding_index'\n  // 使用向量索引进行近似最近邻 (ANN) 搜索，相似度分数需高于阈值\n  CALL db.index.vector.queryNodes('entity_embedding_index', 10, e.embedding)\n  YIELD node, score\n  WITH e, node, score\n  WHERE node.embedding IS NOT NULL\n    AND score > toFloat($cutoff)                 // 筛选高于相似度阈值的节点\n    AND labels(e) = labels(node)                  // 确保候选节点与源节点标签相同\n    AND coalesce(no

,cluster_id,size,members,members_str
0,1061,352,"[Additional Metal Concentration Data, Addition...",Additional Metal Concentration Data | Addition...
1,1158,325,"[Additional Metal Concentration Data, Addition...",Additional Metal Concentration Data | Addition...
2,1286,321,"[Additional Metal Concentration Data, Addition...",Additional Metal Concentration Data | Addition...
3,1287,321,"[Additional Metal Concentration Data, Addition...",Additional Metal Concentration Data | Addition...
4,1192,319,"[Additional Metal Concentration Data, Addition...",Additional Metal Concentration Data | Addition...
...,...,...,...,...
2375,2364,1,[Microbiology of flooded rice paddies],Microbiology of flooded rice paddies
2376,2365,1,[Nitrogen fixation in rice rhizosphere],Nitrogen fixation in rice rhizosphere
2377,2367,1,"[Reciprocal influence of Hg methylation, DOC a...","Reciprocal influence of Hg methylation, DOC an..."
2378,2368,1,[EH and biochar impact on microbial community ...,EH and biochar impact on microbial community a...


***文本描述类实体对齐***

文本描述类的实体包括

* mp_Claim;mp_Statement;
* whu_Bio_chemical_Experiment;
* whu_BioChemicalActivityStep;
* whu_Computational_Experiment;
* whu_ComputationalActivityStep;
* whu_DataSet;
* whu_Specimen_Processing_Activity;
* whu_SpecimenPreprocessing;
* whu_SpecimenCollection;
* whu_Specimen_Collection_Activity;
* whu_Goal;
* whu_Method
 
以上节点合并的思路包括
1. 首先建立similar 关系，这一步实际与fuzzy match 类似，但是这里的similar 关系是基于embedding的，而fuzzy match 是基于字符串的。
2. 通过wcc算法，建立master entity 进行合并，但这里的master entity 中hasname，应该是多个节点的llm综合，而hasoriginaltext属性应该是多个节点的原始文本，应该实习字典形式的[nodeid:originaltext]形式
 

**假设簇为 `[a,b,c]`，融合出新节点 `d`。**

1. “d 保留与 \[a,b,c] 所有关系”

✅ 这是最佳实践。最简单可靠的做法是用 APOC 的 `apoc.refactor.mergeNodes` 并开启 `mergeRels:true`，它会把所有入/出边迁移到保留节点（survivor）上，并去重相同关系。
由于你要新建 `d` 再合并，确保 `d` 放在合并列表第一个位置，这样“保留节点 = d”。

2.  `d.WHU_HASNAME` 由 `[a,b,c]` 的名字经 LLM 归一

✅ 思路很对。生产中常见做法：

* **有 LLM**：用 prompt 让 LLM 选一个“规范名”（canonical name），必要时也输出别名数组。
* **无 LLM / 兜底**：选最长/最常见的变体，或启发式清洗（去括号、去后缀）。
* **建议**：额外加个 `WHU_ALIASES`（列表）存所有去重后的别名，便于后续检索。

3.  `d.WHU_HASORIGINALTEXT` 保存源文本

⚠️ 注意：Neo4j 的单个属性不能直接存 Map；如果你要“字典”，请存成 JSON 字符串（`apoc.convert.toJson()` / Python `json.dumps`），或者更推荐显式的图结构：把 Chunk 做成节点 `(:Chunk)`，用关系 `(:Entity)-[:HAS_CHUNK]->(:Chunk)` 连接（例如 `(d)-[:HAS_CHUNK]->(chunk)`）。
✅ 如果只是数组，直接存 `List<string>` 完全可以。

4. `d.embedding` 用 `[a,b,c]` 中“排序最高”的节点 embedding

✅ 可行但有改进空间：

* **简单**：取向量分数最高那个的 embedding。
* **更优**：把 `d.WHU_HASNAME` 或 `d` 的汇总描述**重新 embed** 一次（你已有 `embed_model`），这样与 `d` 的语义更一致。
  我在代码里提供两种模式：复制 or 重算（可选项）。

5. 可追溯性（强烈建议加）

给 `d` 增加 `sourceElementIds`（`list<string>`）以保留来源节点的 `elementId`；或者把 `[a,b,c]` 不删除，给它们打上 `:Alias` 并连 `(:Alias)-[:MERGED_INTO]->(d)`。
我在代码里采用“直接物理合并”（旧节点删除），并把来源 ids 写进 `d.sourceElementIds`。

## C14 在embedding相似度基础上，建立similary关系
1. 只针对特定node类型（文本描述node）
2. 其中DataSet节点合并结果可能有一定问题，即embedding 接近，但内容差别较大，可能要结合其他方法，比如图论方法进行融合，而不是embeddding方法

In [ ]:
import utilities.return_llm_database as db_manager
import pprint as pp
from typing import List, Dict, Set, Optional

class EmbeddingSimilarityManager:
    """基于embedding向量的同类型节点相似关系构建管理器"""
    
    def __init__(self, remote_database: bool = False):
        """初始化数据库连接和参数配置"""
        self.neo4j_driver = db_manager.DatabaseManager.get_neo4j_driver(
            remotedatebase=remote_database
        )
        self.similarity_threshold = 0.90  # embedding相似度阈值
        self.target_node_types = [
            'mp_Claim', 'mp_Statement', 
            'whu_Bio_chemical_Experiment',
            'whu_BioChemicalActivityStep',
            'whu_Computational_Experiment', 
            'whu_ComputationalActivityStep',
            'whu_DataSet',
            'whu_Specimen_Processing_Activity',
            'whu_SpecimenPreprocessing',
            'whu_SpecimenCollection',
            'whu_Specimen_Collection_Activity',
            'whu_Goal',
            'whu_Method'
        ]
    
    def build_similarity_relations_with_clustering(self) -> dict:
        """
        完整实现代码块A的embedding聚合逻辑 + 代码块B的关系构建方法
        
        Returns:
            dict: 包含聚类结果和关系创建统计信息的字典
        """
        all_results = {}
        
        for node_type in self.target_node_types:
            try:
                print(f"🔍 处理节点类型: {node_type}")
                
                # 步骤1: 使用代码块A的聚合逻辑找出相似节点簇
                similarity_clusters = self._find_similarity_clusters(node_type)
                
                # 步骤2: 基于聚类结果创建SIMILAR关系
                created_relations = self._create_relations_from_clusters(
                    node_type, similarity_clusters
                )
                
                all_results[node_type] = {
                    "clusters_found": len(similarity_clusters),
                    "relations_created": created_relations,
                    "clusters_detail": similarity_clusters
                }
                
                print(f"✓ {node_type}: {len(similarity_clusters)}个聚类, {created_relations}个关系")
                
            except Exception as e:
                all_results[node_type] = {"error": str(e)}
                print(f"✗ {node_type}: 失败 - {str(e)}")
        
        return all_results
    
    def _find_similarity_clusters(self, node_type: str) -> List[List[str]]:
        """
        完全复现代码块A的聚合逻辑，找出相似节点簇
        
        Args:
            node_type: 目标节点类型
            
        Returns:
            List[List[str]]: 相似节点簇列表，每个簇包含节点名称列表
        """
        # 构建完全复现代码块A逻辑的Cypher查询
        cypher_clustering = f"""
        // 完全复现代码块A的聚合逻辑，针对特定节点类型
        MATCH (e:{node_type})
        WHERE e.embedding IS NOT NULL
          AND coalesce(e.WHU_HASNAME, e.WHU_HASORIGINALTEXT, '') <> ''
        
        // 使用子查询为每个实体e寻找前10个最相似的候选节点
        CALL {{
          WITH e
          // 使用向量索引进行近似最近邻 (ANN) 搜索
          CALL db.index.vector.queryNodes('entity_embedding_index', 10, e.embedding)
          YIELD node, score
          WITH e, node, score
          WHERE node.embedding IS NOT NULL
            AND score > toFloat($cutoff)                     // 筛选高于相似度阈值的节点
            AND node:{node_type}                             // 确保候选节点与源节点标签相同
            AND coalesce(node.WHU_HASNAME, node.WHU_HASORIGINALTEXT, '') <> ''
          WITH node, score
          ORDER BY coalesce(node.WHU_HASNAME, node.WHU_HASORIGINALTEXT, '')
          RETURN collect(node) AS nodes                      // 将候选节点收集到列表中
        }}
        
        // 主查询继续
        WITH DISTINCT nodes                                  // 确保候选节点列表唯一
        WHERE size(nodes) > 1                               // 只处理找到多于一个候选节点的簇
        
        // 将每个候选簇中的节点转换为它们的名称，形成一个名称列表
        WITH collect([n IN nodes | coalesce(n.WHU_HASNAME, n.WHU_HASORIGINALTEXT)]) AS results
        
        // 展开结果列表的索引，以便后续两两比较
        UNWIND range(0, size(results)-1, 1) AS index
        WITH results, index, results[index] AS result
        
        // 核心操作：合并具有交集的集合（通过apoc.coll.intersection检测）
        WITH apoc.coll.sort(
          reduce(
            acc = result,                                    // 初始累加器为当前result
            index2 IN range(0, size(results)-1, 1) |        // 遍历所有其他集合
              CASE
                // 如果当前索引不等于遍历索引且两个集合有交集，则合并（取并集）
                WHEN index <> index2 AND size(apoc.coll.intersection(acc, results[index2])) > 0
                  THEN apoc.coll.union(acc, results[index2])
                ELSE acc                                     // 否则保持累加器不变
              END
          )
        ) AS combinedResult
        
        // 对合并后的结果进行去重
        WITH DISTINCT combinedResult
        
        // 进一步去重：过滤掉那些完全被其他更大集合包含的集合（避免冗余）
        WITH collect(combinedResult) AS allCombinedResults
        UNWIND range(0, size(allCombinedResults)-1, 1) AS combinedResultIndex
        WITH allCombinedResults[combinedResultIndex] AS combinedResult, combinedResultIndex, allCombinedResults
        WHERE NOT any(x IN range(0, size(allCombinedResults)-1, 1)    // 检查是否存在另一个集合x
          WHERE x <> combinedResultIndex                              // x不是当前集合
            AND apoc.coll.containsAll(allCombinedResults[x], combinedResult) // 且x完全包含当前集合
        )
        
        // 返回最终去重后的合并结果（每个结果是一个相似实体的名称列表）
        RETURN combinedResult
        """
        
        # 执行聚类查询
        with self.neo4j_driver.session() as session:
            result = session.run(cypher_clustering, {"cutoff": self.similarity_threshold})
            clusters = [record["combinedResult"] for record in result]
            return clusters
    
    def _create_relations_from_clusters(self, node_type: str, clusters: List[List[str]]) -> int:
        """
        基于聚类结果创建SIMILAR关系（采用代码块B的方法）
        
        Args:
            node_type: 节点类型
            clusters: 相似节点簇列表
            
        Returns:
            int: 创建的关系数量
        """
        if not clusters:
            return 0
        
        total_relations = 0
        
        for cluster in clusters:
            if len(cluster) < 2:  # 跳过单节点簇
                continue
            
            # 为簇内每对节点创建SIMILAR关系
            cluster_relations = self._create_relations_within_cluster(node_type, cluster)
            total_relations += cluster_relations
        
        return total_relations
    
    def _create_relations_within_cluster(self, node_type: str, cluster_nodes: List[str]) -> int:
        """
        在单个聚类内部创建两两SIMILAR关系
        
        Args:
            node_type: 节点类型
            cluster_nodes: 聚类中的节点名称列表
            
        Returns:
            int: 创建的关系数量
        """
        # 构建在聚类内创建关系的Cypher查询
        cypher_create_relations = f"""
        // 在聚类内为所有节点对创建SIMILAR关系
        WITH $cluster_nodes as node_names
        UNWIND node_names as name1
        UNWIND node_names as name2
        
        // 找到对应的节点
        MATCH (n1:{node_type}), (n2:{node_type})
        WHERE (coalesce(n1.WHU_HASNAME, n1.WHU_HASORIGINALTEXT, '') = name1)
          AND (coalesce(n2.WHU_HASNAME, n2.WHU_HASORIGINALTEXT, '') = name2)
          AND id(n1) > id(n2)                                // 避免重复关系和自环
          AND NOT (n1)-[:SAME_AS]-(n2)                       // 排除已有SAME_AS关系
        
        // 计算实际的embedding相似度分数（如果可能）
        WITH n1, n2,
             CASE 
               WHEN n1.embedding IS NOT NULL AND n2.embedding IS NOT NULL
               THEN gds.similarity.cosine(n1.embedding, n2.embedding)
               ELSE $default_score
             END as actual_score
        
        // 创建或更新SIMILAR关系
        MERGE (n1)-[r:SIMILAR]->(n2)
        ON CREATE SET
            r.sim_score = actual_score,
            r.embedding_threshold = $similarity_threshold,
            r.createdAt = datetime(),
            r.method = 'embedding_clustering',
            r.cluster_based = true
        ON MATCH SET
            r.lastUpdated = datetime(),
            r.sim_score = CASE WHEN r.sim_score < actual_score THEN actual_score ELSE r.sim_score END
        
        RETURN count(r) as relations_created
        """
        
        # 执行关系创建
        with self.neo4j_driver.session() as session:
            result = session.run(cypher_create_relations, {
                "cluster_nodes": cluster_nodes,
                "similarity_threshold": self.similarity_threshold,
                "default_score": self.similarity_threshold  # 默认分数
            })
            record = result.single()
            return record["relations_created"] if record else 0
    
    def get_clustering_statistics(self) -> dict:
        """
        获取聚类和关系统计信息
        
        Returns:
            dict: 详细的统计信息
        """
        stats = {}
        
        for node_type in self.target_node_types:
            cypher_stats = f"""
            MATCH (n:{node_type})
            OPTIONAL MATCH (n)-[r:SIMILAR {{cluster_based: true}}]-(m:{node_type})
            RETURN 
                count(DISTINCT n) as total_nodes,
                count(DISTINCT CASE WHEN r IS NOT NULL THEN n END) as nodes_in_clusters,
                count(r) as cluster_based_relations,
                count(DISTINCT CASE WHEN n.embedding IS NOT NULL THEN n END) as nodes_with_embedding
            """
            
            with self.neo4j_driver.session() as session:
                result = session.run(cypher_stats)
                record = result.single()
                
                if record:
                    stats[node_type] = {
                        "total_nodes": record["total_nodes"],
                        "nodes_with_embedding": record["nodes_with_embedding"],
                        "nodes_in_clusters": record["nodes_in_clusters"],
                        "cluster_based_relations": record["cluster_based_relations"]
                    }
        
        return stats
    
    def _count_cluster_relations(self, node_type: str) -> int:
        """统计基于聚类的关系数量"""
        cypher = f"""
        MATCH (:{node_type})-[r:SIMILAR {{cluster_based: true}}]-(:{node_type})
        RETURN count(r) as count
        """
        
        with self.neo4j_driver.session() as session:
            result = session.run(cypher)
            record = result.single()
            return record["count"] if record else 0
    
    def close(self):
        """关闭数据库连接"""
        if self.neo4j_driver:
            self.neo4j_driver.close()


# 使用示例
def main():
    """主函数：执行完整的embedding聚类和关系构建流程"""
    # 初始化管理器
    sim_manager = EmbeddingSimilarityManager(remote_database=False)
    
    try:
        print("🚀 开始执行embedding聚类和关系构建...")
        print(f"📊 相似度阈值: {sim_manager.similarity_threshold}")
        print(f"🎯 目标节点类型: {len(sim_manager.target_node_types)}个")
        
        # 执行完整的聚类和关系构建流程
        results = sim_manager.build_similarity_relations_with_clustering()
        
        print("\n📋 聚类和关系构建结果:")
        for node_type, result in results.items():
            if "error" not in result:
                print(f"  {node_type}: {result['clusters_found']}个聚类 → {result['relations_created']}个关系")
            else:
                print(f"  {node_type}: 错误 - {result['error']}")
        
        # 获取详细统计
        print("\n📈 详细统计信息:")
        stats = sim_manager.get_clustering_statistics()
        pp.pprint(stats)

        
    except Exception as e:
        print(f"❌ 执行过程中出现错误: {str(e)}")
    
    finally:
        sim_manager.close()
        print("✅ 执行完成，数据库连接已关闭")


if __name__ == "__main__":
    main()

## C15 结合similar关系与WCC算法，构建master entity节点，实现master entitiy与相似节点之间的reference

1. 与之前概念单位的master entity 不同，文本内容相关的master entity的hasname，hasorignaltext属性应该是多个来源节点的综合，在此设计通过llm实现。

新的版本中，增加master的summary属性，对应多个节点的hasorignaltext 的综合。提供给后期的检索（例如MS GRAPH RAG）使用。
                                         ·

In [5]:
import utilities.return_llm_database as rldb
import importlib

class MasterEntityBuilder:
    """基于WCC算法的Master Entity构建器，支持多节点类型和LLM综合属性生成"""
    
    def __init__(self, remote_database: bool = False):
        """初始化数据库连接和配置"""
        importlib.reload(rldb)
        self.driver = rldb.DatabaseManager.get_neo4j_driver(remotedatebase=remote_database)
        self.llm = rldb.DatabaseManager.get_llm()
        # 目标节点，不是所有节点都进行merge，排除概念类（即有明确的概念名称，例如hg类节点）的融合
        self.target_node_types = [
            'mp_Claim', 'mp_Statement', 
            'whu_Bio_chemical_Experiment',
            'whu_BioChemicalActivityStep',
            'whu_Computational_Experiment', 
            'whu_ComputationalActivityStep',
            'whu_DataSet',
            'whu_Specimen_Processing_Activity',
            'whu_SpecimenPreprocessing',
            'whu_SpecimenCollection',
            'whu_Specimen_Collection_Activity',
            'whu_Goal',
            'whu_Method'
        ]
        self.max_out_degree = 10
    
    def build_all_master_entities(self) -> dict:
        """为所有目标节点类型构建Master Entity"""
        results = {}
        
        for node_type in self.target_node_types:
            try:
                print(f"🔧 处理节点类型: {node_type}")
                result = self._build_master_for_type(node_type)
                results[node_type] = result
                print(f"✓ {node_type}: {result['masters_created']}个Master, {result['references_created']}个引用")
            except Exception as e:
                results[node_type] = {"error": str(e)}
                print(f"✗ {node_type}: 失败 - {str(e)}")
        
        return results
    
    def _build_master_for_type(self, node_type: str) -> dict:
        """为单个节点类型构建Master Entity"""
        graph_name = f"wcc-{node_type.lower()}"
        master_label = f"{node_type}Master"
        
        try:
            # Step 1: 创建图投影
            self._create_graph_projection(graph_name, node_type)
            
            # Step 2: 执行WCC并构建Master节点
            masters_created, references_created = self._build_masters_with_wcc(
                graph_name, node_type, master_label
            )
            
            # Step 3: 使用LLM综合生成Master属性（包括summary）
            self._generate_master_attributes_with_llm(master_label)
            
            # Step 4: 清理高出度Master节点
            cleaned = self._cleanup_high_degree_masters(master_label)
            
            return {
                "masters_created": masters_created,
                "references_created": references_created,
                "cleaned_masters": cleaned
            }
            
        finally:
            # 清理图投影
            self._drop_graph_projection(graph_name)
    
    def _create_graph_projection(self, graph_name: str, node_type: str):
        """创建图投影"""
        # 先尝试删除已存在的图投影
        drop_query = "CALL gds.graph.drop($name, false) YIELD graphName RETURN graphName"
        try:
            with self.driver.session() as session:
                session.run(drop_query, {"name": graph_name}).consume()
        except:
            pass  # 不存在则忽略
        
        # 创建新的图投影
        project_query = f"""
        CALL gds.graph.project(
            $name,
            '{node_type}',
            ['SIMILAR']
        ) 
        YIELD graphName, nodeCount, relationshipCount
        RETURN graphName, nodeCount, relationshipCount
        """
        
        with self.driver.session() as session:
            record = session.run(project_query, {"name": graph_name}).single()
            print(f"  投影创建: {record['nodeCount']}个节点, {record['relationshipCount']}个关系")
    
    def _build_masters_with_wcc(self, graph_name: str, node_type: str, master_label: str) -> tuple:
        """执行WCC并构建Master节点"""
        build_query = f"""
        CALL gds.wcc.stream($graph_name) 
        YIELD nodeId, componentId
        WITH gds.util.asNode(nodeId) AS sourceNode, componentId
        
        // 为每个连通分量创建Master节点
        MERGE (master:{master_label} {{uid: componentId}})
        ON CREATE SET 
            master.createdAt = datetime(),
            master.nodeType = '{node_type}'
        
        // 建立引用关系
        MERGE (master)-[:HAS_REFERENCE]->(sourceNode)
        
        RETURN 
            count(DISTINCT master) as masters_created,
            count(*) as references_created
        """
        
        with self.driver.session() as session:
            result = session.run(build_query, {"graph_name": graph_name}).single()
            return result["masters_created"], result["references_created"]
    
    def _generate_master_attributes_with_llm(self, master_label: str):
        """使用LLM为Master节点生成综合属性（包括summary）"""
        # 获取需要生成属性的Master节点及其引用
        fetch_query = f"""
        MATCH (master:{master_label})-[:HAS_REFERENCE]->(ref)
        WHERE master.hasName IS NULL OR master.originalText IS NULL OR master.summary IS NULL
        WITH master, collect({{
            name: coalesce(ref.WHU_HASNAME, ''),
            text: coalesce(ref.WHU_HASORIGINALTEXT, '')
        }}) as references
        WHERE size([r in references WHERE r.name <> '' OR r.text <> '']) > 0
        RETURN elementId(master) as masterId, references
        LIMIT 50
        """
        
        with self.driver.session() as session:
            records = list(session.run(fetch_query))
            
            for record in records:
                master_id = record["masterId"]
                references = record["references"]
                
                try:
                    # 使用LLM生成综合属性（包括summary）
                    synthesized_attrs = self._synthesize_attributes_with_llm(references)
                    
                    # 更新Master节点属性
                    update_query = f"""
                    MATCH (master:{master_label})
                    WHERE elementId(master) = $master_id
                    SET 
                        master.hasName = $has_name,
                        master.originalText = $original_text,
                        master.summary = $summary,
                        master.synthesizedAt = datetime()
                    """
                    
                    session.run(update_query, {
                        "master_id": master_id,
                        "has_name": synthesized_attrs["name"],
                        "original_text": synthesized_attrs["text"],
                        "summary": synthesized_attrs["summary"]
                    })
                    
                except Exception as e:
                    print(f"  LLM综合失败 Master ID {master_id}: {str(e)}")
                    continue
    
    def _synthesize_attributes_with_llm(self, references: list) -> dict:
        """使用LLM综合多个引用节点的属性"""
        # 构建输入文本
        texts = []
        names = []
        
        for ref in references:
            if ref["text"].strip():
                texts.append(ref["text"].strip())
            if ref["name"].strip():
                names.append(ref["name"].strip())
        
        # 构建LLM提示用于生成name和text
        prompt_attrs = f"""
        请基于以下多个相似实体的文本和名称，生成一个综合的标准化表示：

        文本内容：
        {chr(10).join([f"- {text}" for text in texts[:100]])}  # 限制数量避免过长

        名称：
        {chr(10).join([f"- {name}" for name in names[:10]])}

        请返回JSON格式：
        {{
            "name": "综合标准化名称（简洁明确）",
            "text": "综合标准化文本描述（保留关键信息）"
        }}
        """
        
        # 构建LLM提示用于生成summary
        prompt_summary = f"""You are a helpful assistant responsible for generating a comprehensive summary of the data provided below.
Given one or two entities, and a list of descriptions, all related to the same entity or group of entities.
Please concatenate all of these into a single, comprehensive description. Make sure to include information collected from all the descriptions.
If the provided descriptions are contradictory, please resolve the contradictions and provide a single, coherent summary. Make sure it is written in third person, and include the entity names so we have the full context.

Entity names:
{chr(10).join([f"- {name}" for name in names[:10]])}

Descriptions:
{chr(10).join([f"- {text}" for text in texts[:100]])}

Please provide a comprehensive summary:"""
        
        try:
            import json
            
            # 调用LLM生成name和text
            response_attrs = self.llm.invoke(prompt_attrs)
            if "```json" in response_attrs.content:
                json_str = response_attrs.content.split("```json")[1].split("```")[0].strip()
            else:
                json_str = response_attrs.content.strip()
            
            result_attrs = json.loads(json_str)
            
            # 调用LLM生成summary
            response_summary = self.llm.invoke(prompt_summary)
            summary_text = response_summary.content.strip()
            
            return {
                "name": result_attrs.get("name", names[0] if names else "")[:500],
                "text": result_attrs.get("text", texts[0] if texts else "")[:1000],
                "summary": summary_text[:2000]  # 限制summary长度
            }
            
        except Exception:
            # LLM失败时的fallback策略
            return {
                "name": names[0] if names else "",
                "text": texts[0] if texts else "",
                "summary": texts[0] if texts else ""  # fallback使用第一个text作为summary
            }
    
    def _cleanup_high_degree_masters(self, master_label: str) -> int:
        """清理高出度的Master节点"""
        cleanup_query = f"""
        MATCH (master:{master_label})
        WHERE COUNT {{ (master)-[:HAS_REFERENCE]->() }} > $max_degree
        DETACH DELETE master
        RETURN count(master) as deleted_count
        """
        
        with self.driver.session() as session:
            result = session.run(cleanup_query, {"max_degree": self.max_out_degree}).single()
            deleted = result["deleted_count"] if result else 0
            if deleted > 0:
                print(f"  清理了{deleted}个高出度Master节点")
            return deleted
    
    def _drop_graph_projection(self, graph_name: str):
        """删除图投影释放内存"""
        try:
            with self.driver.session() as session:
                session.run("CALL gds.graph.drop($name, false)", {"name": graph_name}).consume()
        except:
            pass  # 忽略删除失败
    
    def get_master_statistics(self) -> dict:
        """获取Master节点统计信息"""
        stats = {}
        
        for node_type in self.target_node_types:
            master_label = f"{node_type}_Master"
            
            stats_query = f"""
            MATCH (master:{master_label})
            OPTIONAL MATCH (master)-[:HAS_REFERENCE]->(ref)
            RETURN 
                count(DISTINCT master) as total_masters,
                count(ref) as total_references,
                count(CASE WHEN master.hasName IS NOT NULL THEN 1 END) as masters_with_name,
                count(CASE WHEN master.synthesizedAt IS NOT NULL THEN 1 END) as llm_synthesized,
                count(CASE WHEN master.summary IS NOT NULL THEN 1 END) as masters_with_summary
            """
            
            with self.driver.session() as session:
                result = session.run(stats_query).single()
                if result and result["total_masters"] > 0:
                    stats[node_type] = {
                        "total_masters": result["total_masters"],
                        "total_references": result["total_references"],
                        "masters_with_name": result["masters_with_name"],
                        "llm_synthesized": result["llm_synthesized"],
                        "masters_with_summary": result["masters_with_summary"]
                    }
        
        return stats
    
    def close(self):
        """关闭数据库连接"""
        if self.driver:
            self.driver.close()


def main():
    """主函数：执行Master Entity构建流程"""
    builder = MasterEntityBuilder(remote_database=False)
    
    try:
        print("🚀 开始构建Master Entity...")
        
        # 构建所有类型的Master Entity
        results = builder.build_all_master_entities()
        
        print("\n📋 构建结果汇总:")
        for node_type, result in results.items():
            if "error" not in result:
                print(f"  {node_type}: {result['masters_created']}个Master, "
                      f"{result['references_created']}个引用, "
                      f"{result['cleaned_masters']}个清理")
            else:
                print(f"  {node_type}: 错误 - {result['error']}")
        
        # 获取统计信息
        print("\n📊 Master Entity统计:")
        stats = builder.get_master_statistics()
        for node_type, stat in stats.items():
            print(f"  {node_type}: {stat['total_masters']}个Master节点, "
                  f"{stat['total_references']}个引用, "
                  f"{stat['llm_synthesized']}个LLM综合, "
                  f"{stat['masters_with_summary']}个带summary")
        
    except Exception as e:
        print(f"❌ 执行失败: {str(e)}")
    
    finally:
        builder.close()
        print("✅ 执行完成")


if __name__ == "__main__":
    main()

🚀 开始构建Master Entity...
🔧 处理节点类型: mp_Claim
  投影创建: 2983个节点, 86506个关系
  清理了35个高出度Master节点


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('schema' returned by 'gds.graph.drop' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: 'CALL gds.graph.drop($name, false)'


✓ mp_Claim: 698个Master, 2983个引用
🔧 处理节点类型: mp_Statement
  投影创建: 2977个节点, 12110个关系


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('schema' returned by 'gds.graph.drop' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: 'CALL gds.graph.drop($name, false)'


  清理了26个高出度Master节点
✓ mp_Statement: 1140个Master, 2977个引用
🔧 处理节点类型: whu_Bio_chemical_Experiment
  投影创建: 320个节点, 503个关系


...

✓ whu_Goal: 143个Master, 192个引用
🔧 处理节点类型: whu_Method
  投影创建: 1079个节点, 3060个关系


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('schema' returned by 'gds.graph.drop' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: 'CALL gds.graph.drop($name, false)'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownLabelWarning} {category: UNRECOGNIZED} {title: The provided label is not in the database.} {description: One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing label name is: mp_Claim_Master)} {position: line: 2, column: 27, offset: 27} for query: '\n            MATCH (master:mp_Claim_Master)\n            OPTIONAL MATCH (master)-[:

  清理了14个高出度Master节点
✓ whu_Method: 485个Master, 1079个引用

📋 构建结果汇总:
  mp_Claim: 698个Master, 2983个引用, 35个清理
  mp_Statement: 1140个Master, 2977个引用, 26个清理
  whu_Bio_chemical_Experiment: 141个Master, 320个引用, 2个清理
  whu_BioChemicalActivityStep: 307个Master, 1166个引用, 25个清理
  whu_Computational_Experiment: 142个Master, 287个引用, 2个清理
  whu_ComputationalActivityStep: 238个Master, 616个引用, 7个清理
  whu_DataSet: 654个Master, 3036个引用, 39个清理
  whu_Specimen_Processing_Activity: 162个Master, 692个引用, 10个清理
  whu_SpecimenPreprocessing: 185个Master, 408个引用, 1个清理
  whu_SpecimenCollection: 75个Master, 165个引用, 1个清理
  whu_Specimen_Collection_Activity: 106个Master, 470个引用, 4个清理
  whu_Goal: 143个Master, 192个引用, 0个清理
  whu_Method: 485个Master, 1079个引用, 14个清理

📊 Master Entity统计:


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownLabelWarning} {category: UNRECOGNIZED} {title: The provided label is not in the database.} {description: One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing label name is: mp_Statement_Master)} {position: line: 2, column: 27, offset: 27} for query: '\n            MATCH (master:mp_Statement_Master)\n            OPTIONAL MATCH (master)-[:HAS_REFERENCE]->(ref)\n            RETURN \n                count(DISTINCT master) as total_masters,\n                count(ref) as total_references,\n                count(CASE WHEN master.hasName IS NOT NULL THEN 1 END) as masters_with_name,\n                count(CASE WHEN master.synthesizedAt IS NOT NULL THEN 1 END) as llm_synthesized,\n                count(CASE WHEN master.summary IS NOT NULL THEN 1 END) as

✅ 执行完成


### C 15.2 为Master 节点添加label ”__Master__”

In [6]:
from utilities import return_llm_database as rldb
if __name__ == "__main__":
    query = "MATCH (n)-[:HAS_REFERENCE]->() SET n:__Master__ RETURN count(n) AS nodesLabeled"
    neo4j_driver = rldb.DatabaseManager.get_neo4j_driver(remotedatebase=False)
    with neo4j_driver.session() as session:
        result = session.run(query)
        count = result.single()["nodesLabeled"]
        print(f"标记为 __Master__ 的节点数量: {count}")



标记为 __Master__ 的节点数量: 9771


### C15.3 实现关系的合并与summary

参考<Essetial GraphRag> Chapter 7 ， 提到在抽取关系后，将nodes之间多个relation进行合并，同时根据关系原有的text，生成summary

但本研究中，最初的抽取并未保留relation的text，在实现时，可以让程序判断如果存在text则生成summary，否则只简单地根据relation的type生成summary。

>实际上，在之前的Master节点生成时，就使用了summary 方法，不过与书中不同，不是每一个entity都具有相同的summary，而是对应到共引的__Master__上，通过__Master__的summary,返回给检索。

----

To avoid inconsistencies, redundancies, and fragmentation in the extracted knowledge, MS GraphRAG merges multiple descriptions of the same entity or relationship
using LLMs to generate concise summaries. Instead of treating each description separately, the model synthesizes information from all descriptions, ensuring that key contextual details are preserved in a single, enriched representation. This approach
enhances clarity, reduces duplication, and provides a more complete understanding
of entities and their relationships. 
 Once again, you can reuse the summarization prompt from the paper, as shown in
“Instructions for entity and relationship summarization.”

----


同时这里需要注意的是，书中的RELATION 是非语义抽取得到的临时relation，估对其合并与summary可以认为必须，但本研究中，relations是基于预定义schema，但对其进行合并和summary依然有其意义。

您提出的疑问非常深入，它直接指出了两种知识图谱（KG）构建策略在数据精炼阶段的不同目标。

**结论：**

对于您这种**基于预定义 Schema 的关系定义方法**，对多关系进行合并和摘要（Summarization）是**有意义的，但其必要性不如 Microsoft GraphRAG (MS GraphRAG) 那么高**。这种操作的目的从“关系类型规范化”转变为“**知识去冗余和增强事实一致性**”。

以下是基于来源资料的详细分析和解释：

1. 预定义 Schema 方法中的关系特点

在采用的 Schema-First（Schema 优先）方法中，LLM 在提取数据时，已经被明确指示必须使用预定义的、具有清晰语义的关系类型（例如，在法律合同中是 `:HAS_PARTY`，在电影图谱中是 `:DIRECTED`）。

*   **关系类型已规范化：** LLM 的输出严格遵循您定义的本体论（Ontologies）或分类法（Taxonomies），确保了语义的一致性。
*   **提取目标清晰：** 目标是将非结构化信息映射到预定义的结构化字段中。
*   **挑战焦点：** 这种方法的主要挑战在于 **Text-to-Cypher** 阶段的准确性（LLM 如何将用户查询准确映射到预定义的 Schema）以及 **实体解析**（Entity Resolution，将不同文本中的实体指称解析到同一个规范化节点）。

2. 多关系合并和摘要的意义

即使关系类型是预定义的，当知识图谱从大量、分散的文本中构建时（尤其是当文本被切分成多个块进行处理时），同一实体对之间仍然可能产生多条语义重复或互补的**关系实例 (Relationship Instances)**。

在这些情况下，关系合并和摘要仍然具有关键作用：

* A. 消除碎片化和冗余（De-fragmentation and Redundancy）

  *   **场景：** 无论关系是否是预定义的，如果文本被切割成块（Chunking），同一个事实可能在不同块中被提及，导致多次提取。
      *   例如，在法律文档中，同一份合同的条款可能分散在多个文本块中。如果 LLM 在处理每个块时都提取了 `Organization A -[:HAS_PARTY]-> Contract B` 这个关系，那么数据库中可能会出现多个连接 A 和 B 的关系实例。
  *   **意义：** 即使关系类型相同，通过 **LLM 摘要**可以合并这些重复的或细微差异的证据（例如，关系上的描述属性），将其精炼为一个单一、连贯、非冗余的表示。这确保了您的 KG 层是**精炼和规范化**的（类似于 MS GraphRAG 将原始关系总结为 `:SUMMARIZED_RELATIONSHIP`）。

* B. 整合上下文证据和属性（Consolidating Evidence and Context）

  *   **场景：** 如果您的预定义 Schema 允许在关系上存储丰富的属性或文本描述（例如，记录关系的来源、日期、或 LLM 提取的总结性描述），那么同一关系实例的多次提取可能提供互补的上下文信息。
  *   **意义：** 关系摘要可以收集所有这些描述（`description_list`），并利用 LLM 的生成能力将它们融合成一个**综合性描述**。这对于提高信息的可信度和可解释性至关重要，特别是当您需要追溯该关系的所有原始证据时。

* C. 维护数据完整性（Data Integrity in Multi-Source Integration）

  *   在构建知识图谱时，通常需要整合来自多个异构数据源的信息。
  *   如果两个不同的结构化数据源（例如，HMDD 和 dbDEMC，在 MiRNA 示例中）都表明 **MiRNA A** 和 **Disease B** 之间存在关联，即使您将关系命名为 `:ASSOCIATED_WITH`，您也需要一个机制来**合并**这两条关系实例（如果它们指代的是相同的语义连接），或者至少通过属性来**记录它们的不同来源和证据**，并可能对冲突信息进行总结和调和。

总结

虽然您的 Schema-First 方法减少了 MS GraphRAG 必须处理的 **“关系类型”的语义混乱**，但它并不能消除**“关系实例”的冗余和碎片化**，尤其是在处理大规模和分块的非结构化数据时。

因此，对预定义关系进行合并和摘要：

1.  **保证了 KG 作为“单一事实来源”的清晰度**。
2.  **提高了检索效率**，避免在查询时（例如通过局部搜索）检索到多个重复的关系描述。
3.  **增强了可解释性**，提供了一个由 LLM 生成的、整合了所有原始证据的权威性关系总结。
   

----

对应 cypher

```cypher 
MATCH (s:__Entity__)-[r]-(t:__Entity__)
WHERE id(s) < id(t)
WITH s.WHU_HASNAME AS source, t.WHU_HASNAME AS target,
     type(r) AS relationship_type, // 新增：获取关系类型
     r.description AS description, // 修改：不再使用collect，暂保留单条描述
     count(*) AS count
WHERE count > 1
RETURN source, target, relationship_type, description // 修改：返回关系类型和描述
```

这里存在问题在于，由于本项目是基于scheam进行抽取，其关系已经比较明确，同时初始抽取时，relation并没有保存text文本，即便两个节点之间同一个类型relation 在原始文本中有重复，也无法区分。

同时，通过检索发现，同一对节点之间存在不同类型的关系情况很少，即便有也不太能说明问题。

故此书中的情况与本项目情况不同，本项目中，可以考虑两种做法：
1. 为每一个relation实例创建一个summary，来源于同一类type 的relation 示例的summary，例如a 节点和b节点之间存在多个来源于文献的关系，首先通过HASORIGINALTEXT 以列表形式保存这些关系的原始文本。

再通过llm对这些关系的text进行summary。但从通用性而言，这可能涉及如果a，b节点存在不同类型的关系，例如a到b存在关系r1，同时存在关系r2，这两种关系的summary可能不同， a，b节点之间缺少对r1，r2关系的综合大的summary。

2. 可以在第一种方法上，进一步将a，b之间的不同类型r1，r2 进行基于的summary，得到新的关系 SUMMARIED_RELATIONN 对两个节点之间关系进行进一步的总结。



In [ ]:
import tqdm


query="""
MATCH (s:__Entity__)-[r]-(t:__Entity__)
WHERE id(s) < id(t)
WITH s.WHU_HASNAME AS source, t.WHU_HASNAME AS target,
     type(r) AS relationship_type, // 新增：获取关系类型
     r.description AS description, // 修改：不再使用collect，暂保留单条描述
     count(*) AS count
WHERE count > 1
RETURN source, target, relationship_type, description // 修改：返回关系类型和描述
"""



with neo4j_driver.session() as session:
        rels_to_summarize = session.run(query)
 

SUMMARIZE_PROMPT = """
You are a helpful assistant responsible for generating a comprehensive summary of the data provided below.
Given one or two entities, and a list of descriptions, all related to the same entity or group of entities.
Please concatenate all of these into a single, comprehensive description. Make sure to include information collected from all the descriptions.
If the provided descriptions are contradictory, please resolve the contradictions and provide a single, coherent summary.
Make sure it is written in third person, and include the entity names so we have the full context.
---
if the description list is empty, summary the relation base on the type and the number of relations.

#######
-Data-
Entities: {entity_name}
number of relations: {num_relations}
Description List: {description_list}
#######
Output:
"""

def import_rels_summary(neo4j_driver, rel_summaries):
    neo4j_driver.execute_query("""
    UNWIND $data AS row
    MATCH (s:__Entity__ {name: row.source}), (t:__Entity__ {name: row.target})
    MERGE (s)-[r:SUMMARIZED_RELATIONSHIP]-(t)
    SET r.summary = row.summary
    """, data=rel_summaries)
    
    # If there was only 1 description use that
    neo4j_driver.execute_query("""
    MATCH (s:__Entity__)-[e:RELATIONSHIP]-(t:__Entity__)
    WHERE NOT (s)-[:SUMMARIZED_RELATIONSHIP]-(t)
    MERGE (s)-[r:SUMMARIZED_RELATIONSHIP]-(t)
    SET r.summary = e.description
    """)

def get_summarize_prompt(entity_name, description_list):
    return SUMMARIZE_PROMPT.format(
        entity_name=entity_name,
        description_list=description_list)


if __name__ == "__main__":
    rel_summaries = []
    for candidate in tqdm(rels_to_summarize, desc="Summarizing relationships"):
        entity_name = f"{candidate['source']} relationship to 
        {candidate['target']}"
        
        messages = [
        {
        "role": "user",
        "content": get_summarize_prompt(
        entity_name, candidate["description_list"]
        ),
        },
        ]
    
    summary = chat(messages, model="gpt-4o")
    rel_summaries.append({"source": candidate["source"], "target": 
    candidate["target"], "summary": summary}) 
    ch07_tools.import_rels_summary(neo4j_driver, summaries)

C15.2 构建Master relation
也合并了之前relation的text

In [9]:
import logging
from utilities import return_llm_database as rldb

# 配置日志，方便调试
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class SEGBuilder:
    def __init__(self, threshold=0.6):
        """
        初始化 SEG 构建器
        :param threshold: llm_weight 的过滤阈值，低于此分数的原关系将不会参与 Master 关系的构建
        """
        self.driver = rldb.DatabaseManager.get_neo4j_driver(remotedatebase=False)
        self.threshold = threshold

    def close(self):
        if self.driver:
            self.driver.close()

    def clear_existing_master_rels(self):
        """
        清理现有的 Master 关系，保证构建的幂等性（可重复运行）
        """
        query = """
        CALL apoc.periodic.iterate(
            "MATCH (:__Master__)-[r]->(:__Master__) WHERE type(r) STARTS WITH 'MASTER_' RETURN r",
            "DELETE r",
            {batchSize: 1000}
        )
        """
        try:
            with self.driver.session() as session:
                logger.info("正在清理旧的 Master 关系...")
                session.run(query)
                logger.info("清理完成。")
        except Exception as e:
            logger.error(f"清理失败: {e}")

   
        """
        核心方法：利用 APOC 构建动态类型的 Master 关系
        包含：属性合并、加权打分、阈值过滤、来源溯源 ID 记录
        """
        # Cypher 逻辑优化亮点：
        # 1. 使用 elementId(r) 记录来源，确保在 Neo4j 5.x 环境下的稳定性
        # 2. 使用 apoc.text.join 替代 reduce，提高长文本拼接效率
        # 3. 增加 from_relation_ids 数组属性，实现完整的数据溯源链
        
        cypher_query = """
        CALL apoc.periodic.iterate(
            "
            MATCH (m1:__Master__)-[:HAS_REFERENCE]->(o1)-[r]->(o2)<-[:HAS_REFERENCE]-(m2:__Master__)
            WHERE m1 <> m2 
              AND r.llm_weight >= $threshold
            RETURN m1, m2, type(r) AS origType, collect(r) AS rels
            ",
            "
            // --- 1. 数据处理与计算 ---
            WITH m1, m2, origType, rels,
                 size(rels) AS support_count,
                 
                 // 提取来源关系的 elementId 列表 (用于溯源)
                 [rel in rels | elementId(rel)] AS source_ids,
                 
                 // 计算基于 LLM 置信度的综合分数
                 avg([rel in rels | COALESCE(rel.llm_weight, 0.5)]) AS avg_confidence,
                 
                 // 高效合并属性文本
                 apoc.text.join([rel in rels | COALESCE(rel.WHU_HASORIGINALTEXT, '')], ' || ') AS merged_text,
                 apoc.text.join([rel in rels | COALESCE(rel.WHU_HASNAME, '')], '; ') AS merged_names

            // --- 2. 动态创建 Master 关系 ---
            WITH m1, m2, 'MASTER_' + origType AS newRelType, 
                 merged_text, merged_names, avg_confidence, support_count, source_ids
            
            CALL apoc.create.relationship(m1, newRelType, {
                WHU_HASORIGINALTEXT: merged_text,
                WHU_HASNAME: merged_names,
                
                // 溯源核心属性
                from_relation_ids: source_ids,  // 记录所有来源关系的 ID
                
                // 核心指标
                master_score: avg_confidence,   // 综合置信度
                support_count: support_count,   // 底层证据支撑数
                
                // 元数据
                label_tag: '__Master__',
                is_derived: true,
                threshold_used: $threshold,
                created_at: datetime()
            }, m2) YIELD rel
            RETURN count(rel)
            ",
            {batchSize: 500, parallel: false, params: {threshold: $threshold}}
        )
        """
        
        try:
            with self.driver.session() as session:
                logger.info(f"开始构建 Master 关系 (阈值: {self.threshold})...")
                result = session.run(cypher_query, threshold=self.threshold)
                summary = result.single()
                logger.info(f"Master 关系构建完成。批次处理详情: {summary}")
        except Exception as e:
            logger.error(f"构建过程中发生错误: {e}")
            raise e
    def build_dynamic_master_relations(self):
        """
        核心方法：修复了 avg() 列表类型不匹配问题的版本
        """
        cypher_query = """
        CALL apoc.periodic.iterate(
            "
            MATCH (m1:__Master__)-[:HAS_REFERENCE]->(o1)-[r]->(o2)<-[:HAS_REFERENCE]-(m2:__Master__)
            WHERE m1 <> m2 
              AND r.llm_weight >= $threshold
            RETURN m1, m2, type(r) AS origType, collect(r) AS rels
            ",
            "
            // --- 1. 数据处理与计算 ---
            WITH m1, m2, origType, rels,
                 size(rels) AS support_count,
                 [rel in rels | elementId(rel)] AS source_ids,
                 
                 // 【修复点】使用 apoc.coll.avg 处理列表求均值
                 apoc.coll.avg([rel in rels | COALESCE(rel.llm_weight, 0.5)]) AS avg_confidence,
                 
                 apoc.text.join([rel in rels | COALESCE(rel.WHU_HASORIGINALTEXT, '')], ' || ') AS merged_text,
                 apoc.text.join([rel in rels | COALESCE(rel.WHU_HASNAME, '')], '; ') AS merged_names

            // --- 2. 动态创建 Master 关系 ---
            WITH m1, m2, 'MASTER_' + origType AS newRelType, 
                 merged_text, merged_names, avg_confidence, support_count, source_ids
            
            CALL apoc.create.relationship(m1, newRelType, {
                WHU_HASORIGINALTEXT: merged_text,
                WHU_HASNAME: merged_names,
                from_relation_ids: source_ids,
                master_score: avg_confidence,
                support_count: support_count,
                label_tag: '__Master__',
                is_derived: true,
                threshold_used: $threshold,
                created_at: datetime()
            }, m2) YIELD rel
            RETURN count(rel)
            ",
            {batchSize: 500, parallel: false, params: {threshold: $threshold}}
        )
        """
        try:
            with self.driver.session() as session:
                logger.info(f"开始构建 Master 关系 (阈值: {self.threshold})...")
                result = session.run(cypher_query, threshold=self.threshold)
                summary = result.single()
                
                # 检查 APOC 执行是否有内部错误
                if summary.get('failedOperations') > 0:
                    logger.error(f"构建过程中存在失败的操作: {summary.get('errorMessages')}")
                else:
                    logger.info(f"Master 关系构建完成。批次处理详情: {summary}")
        except Exception as e:
            logger.error(f"构建过程中发生网络或连接错误: {e}")
            raise e
    def verify_results(self):
        """验证构建结果及溯源属性"""
        query = """
        MATCH (n:__Master__)-[r]->(m:__Master__) 
        WHERE type(r) STARTS WITH 'MASTER_'
        RETURN type(r) as Type, 
               count(r) as Count, 
               avg(r.master_score) as AvgScore,
               r.from_relation_ids[0] as SampleSourceID
        LIMIT 5
        """
        with self.driver.session() as session:
            result = session.run(query)
            print("\n--- 构建结果与溯源验证 ---")
            print(f"{'关系类型':<25} | {'数量':<8} | {'平均分数':<10} | {'样本来源ID'}")
            print("-" * 85)
            for record in result:
                print(f"{record['Type']:<25} | {record['Count']:<8} | {record['AvgScore']:.4f} | {record['SampleSourceID']}")

if __name__ == "__main__":
    builder = SEGBuilder(threshold=0)
    try:
        builder.clear_existing_master_rels()
        builder.build_dynamic_master_relations()
        builder.verify_results()
    finally:
        builder.close()

2025-12-16 15:24:09,315 - INFO - 正在清理旧的 Master 关系...
2025-12-16 15:24:09,329 - INFO - 清理完成。
2025-12-16 15:24:09,339 - INFO - 开始构建 Master 关系 (阈值: 0)...
2025-12-16 15:24:09,606 - INFO - Master 关系构建完成。批次处理详情: <Record batches=4 total=1744 timeTaken=0 committedOperations=0 failedOperations=1744 failedBatches=4 retries=0 errorMessages={'Type mismatch: expected Float, Integer or Duration but was List<T> (line 10, column 22 (offset: 414))\r\n"                 avg([rel in rels | COALESCE(rel.llm_weight, 0.5)]) AS avg_confidence,"\r\n                      ^': 4} batch={'total': 4, 'errors': {'org.neo4j.graphdb.QueryExecutionException: Type mismatch: expected Float, Integer or Duration but was List<T> (line 10, column 22 (offset: 414))\r\n"                 avg([rel in rels | COALESCE(rel.llm_weight, 0.5)]) AS avg_confidence,"\r\n                      ^': 4}, 'committed': 0, 'failed': 4} operations={'total': 1744, 'errors': {'Type mismatch: expected Float, Integer or Duration but was List<T> (line


--- 构建结果与溯源验证 ---
关系类型                      | 数量       | 平均分数       | 样本来源ID
-------------------------------------------------------------------------------------
MASTER_whu_hasActivity    | 1        | 0.7000 | 5:a588b8f8-4a1b-4a79-8a1a-924958fc2820:1157474582257485369
MASTER_mp_challenges      | 1        | 0.7500 | 5:a588b8f8-4a1b-4a79-8a1a-924958fc2820:1152968783606861257
MASTER_mp_challenges      | 1        | 0.9000 | 5:a588b8f8-4a1b-4a79-8a1a-924958fc2820:1152968783606844258
MASTER_mp_challenges      | 1        | 0.8500 | 5:a588b8f8-4a1b-4a79-8a1a-924958fc2820:1152968783606841512
MASTER_mp_challenges      | 1        | 0.9000 | 5:a588b8f8-4a1b-4a79-8a1a-924958fc2820:1152968783606841513


## 1.5.3 为Master node,Master relation 构建embedding索引
注意  Master node，Master relation实际构成了SEG子图，为了后续的检索，需要为其构建embedding索引

In [ ]:
import utilities.return_llm_database
import importlib
import torch
importlib.reload(utilities.return_llm_database)
 
embed = utilities.return_llm_database.DatabaseManager()

In [ ]:
import utilities.return_llm_database
import importlib
import torch
importlib.reload(utilities.return_llm_database)

class MasterEmbeddingManager:
    """Master节点和关系的Embedding管理器"""
    
    def __init__(self, expected_dimension=None):
        """初始化管理器"""
        print("="*70)
        print("初始化 Master Embedding Manager")
        print("="*70)
        
        manager = utilities.return_llm_database.DatabaseManager()
        self.llm, self.embed_model, self.neo4j_driver = manager.get_components()
        
        self.expected_dimension = expected_dimension
        self._check_gpu_environment()
        self._detect_embedding_dimension()
        
        print("初始化完成\n")
    
    def _detect_embedding_dimension(self):
        """检测embedding模型的实际维度"""
        try:
            if hasattr(self.embed_model, 'embed_query'):
                test_embedding = self.embed_model.embed_query("测试")
            elif hasattr(self.embed_model, 'embed_documents'):
                test_embedding = self.embed_model.embed_documents(["测试"])[0]
            else:
                test_embedding = self.embed_model("测试")
            
            if hasattr(test_embedding, 'tolist'):
                test_embedding = test_embedding.tolist()
            elif torch.is_tensor(test_embedding):
                test_embedding = test_embedding.cpu().tolist() if test_embedding.is_cuda else test_embedding.tolist()
            
            actual_dim = len(test_embedding)
            print(f"模型实际维度: {actual_dim}")
            
            if self.expected_dimension is None:
                self.expected_dimension = actual_dim
            elif self.expected_dimension != actual_dim:
                print(f"⚠️  维度不匹配: 期望{self.expected_dimension} != 实际{actual_dim}")
                self.expected_dimension = actual_dim
                
        except Exception as e:
            print(f"检测维度失败: {e}")
            self.expected_dimension = self.expected_dimension or 768
    
    def _check_gpu_environment(self):
        """检查GPU环境"""
        if torch.cuda.is_available():
            self.device = 'cuda'
            print(f"GPU可用: {torch.cuda.get_device_name(0)}")
            torch.cuda.empty_cache()
        else:
            self.device = 'cpu'
            print("使用CPU")
    
    def get_master_nodes_for_embedding(self, limit=100):
        """获取需要创建embedding的Master节点"""
        query = """
        MATCH (n:__Master__)
        WHERE (n.WHU_HASNAME IS NOT NULL AND trim(toString(n.WHU_HASNAME)) <> ''
               OR n.WHU_HASORIGINALTEXT IS NOT NULL AND trim(toString(n.WHU_HASORIGINALTEXT)) <> ''
               OR n.summary IS NOT NULL AND trim(toString(n.summary)) <> '')
               AND n.embedding IS NULL
        RETURN elementId(n) as id, 
               n.WHU_HASNAME as name,
               n.WHU_HASORIGINALTEXT as text,
               n.summary as summary,
               labels(n) as labels
        LIMIT $limit
        """
        
        with self.neo4j_driver.session() as session:
            try:
                result = session.run(query, limit=limit)
                records = list(result)
                print(f"找到 {len(records)} 个待处理的Master节点")
                return records
            except Exception as e:
                print(f"获取Master节点失败: {e}")
                return []
    
    def get_master_relations_for_embedding(self, limit=100):
        """获取需要创建embedding的Master关系"""
        query = """
        MATCH (n)-[r]->(m)
        WHERE r.label_tag = '__Master__'
              AND (r.WHU_HASNAME IS NOT NULL AND trim(toString(r.WHU_HASNAME)) <> ''
                   OR r.WHU_HASORIGINALTEXT IS NOT NULL AND trim(toString(r.WHU_HASORIGINALTEXT)) <> '')
              AND r.embedding IS NULL
        RETURN elementId(r) as id,
               type(r) as rel_type,
               r.WHU_HASNAME as name,
               r.WHU_HASORIGINALTEXT as text,
               n.WHU_HASNAME as source_name,
               m.WHU_HASNAME as target_name
        LIMIT $limit
        """
        
        with self.neo4j_driver.session() as session:
            try:
                result = session.run(query, limit=limit)
                records = list(result)
                print(f"找到 {len(records)} 个待处理的Master关系")
                return records
            except Exception as e:
                print(f"获取Master关系失败: {e}")
                return []
    
    def create_text_for_node_embedding(self, node):
        """为Master节点创建embedding文本（优先级：summary > name > text）"""
        summary = str(node.get('summary') or "").strip()
        name = str(node.get('name') or "").strip()
        text = str(node.get('text') or "").strip()
        
        parts = []
        if summary and summary not in ["None", ""]:
            parts.append(summary)
        if name and name not in ["None", ""]:
            parts.append(name)
        if text and text not in ["None", ""] and text != name:
            parts.append(text)
        
        return " | ".join(parts) if parts else None
    
    def create_text_for_relation_embedding(self, relation):
        """为Master关系创建embedding文本"""
        rel_type = str(relation.get('rel_type') or "").strip()
        name = str(relation.get('name') or "").strip()
        text = str(relation.get('text') or "").strip()
        source = str(relation.get('source_name') or "").strip()
        target = str(relation.get('target_name') or "").strip()
        
        parts = []
        if source:
            parts.append(f"源:{source}")
        if rel_type:
            parts.append(f"关系:{rel_type}")
        if target:
            parts.append(f"目标:{target}")
        if name and name not in ["None", ""]:
            parts.append(name)
        if text and text not in ["None", ""] and text != name:
            parts.append(text)
        
        return " | ".join(parts) if parts else None
    
    def generate_embeddings_batch(self, texts):
        """批量生成embeddings（与源代码逻辑一致）"""
        if not texts:
            return []
        
        try:
            if hasattr(self.embed_model, 'embed_documents'):
                embeddings = self.embed_model.embed_documents(texts)
            elif hasattr(self.embed_model, 'embed_query'):
                embeddings = [self.embed_model.embed_query(text) for text in texts]
            else:
                embeddings = [self.embed_model(text) for text in texts]
            
            processed = []
            for emb in embeddings:
                if hasattr(emb, 'tolist'):
                    processed.append(emb.tolist())
                elif torch.is_tensor(emb):
                    processed.append(emb.cpu().tolist() if emb.is_cuda else emb.tolist())
                else:
                    processed.append(emb)
            
            return processed
            
        except Exception as e:
            print(f"批量生成失败，回退到逐个处理: {e}")
            return self._generate_embeddings_one_by_one(texts)
    
    def _generate_embeddings_one_by_one(self, texts):
        """逐个生成embedding"""
        embeddings = []
        for i, text in enumerate(texts):
            try:
                if hasattr(self.embed_model, 'embed_query'):
                    emb = self.embed_model.embed_query(text)
                elif hasattr(self.embed_model, 'embed_documents'):
                    emb = self.embed_model.embed_documents([text])[0]
                else:
                    emb = self.embed_model(text)
                
                if hasattr(emb, 'tolist'):
                    emb = emb.tolist()
                elif torch.is_tensor(emb):
                    emb = emb.cpu().tolist() if emb.is_cuda else emb.tolist()
                
                embeddings.append(emb)
                
                if (i + 1) % 10 == 0:
                    print(f"  已处理 {i + 1}/{len(texts)}")
            except Exception as e:
                print(f"  处理失败: {text[:30]}... | 错误: {e}")
        
        return embeddings
    
    def update_node_embeddings(self, node_ids, embeddings):
        """批量更新节点embeddings"""
        if len(node_ids) != len(embeddings):
            print(f"长度不匹配: {len(node_ids)} vs {len(embeddings)}")
            return 0
        
        success_count = 0
        with self.neo4j_driver.session() as session:
            for node_id, embedding in zip(node_ids, embeddings):
                try:
                    if len(embedding) != self.expected_dimension:
                        print(f"⚠️  维度错误: 期望{self.expected_dimension}, 实际{len(embedding)}")
                        continue
                    
                    session.run("""
                        MATCH (n) WHERE elementId(n) = $node_id
                        SET n.embedding = $embedding
                    """, node_id=node_id, embedding=embedding)
                    success_count += 1
                except Exception as e:
                    print(f"更新节点失败: {e}")
        
        return success_count
    
    def update_relation_embeddings(self, rel_ids, embeddings):
        """批量更新关系embeddings"""
        if len(rel_ids) != len(embeddings):
            print(f"长度不匹配: {len(rel_ids)} vs {len(embeddings)}")
            return 0
        
        success_count = 0
        with self.neo4j_driver.session() as session:
            for rel_id, embedding in zip(rel_ids, embeddings):
                try:
                    if len(embedding) != self.expected_dimension:
                        print(f"⚠️  维度错误: 期望{self.expected_dimension}, 实际{len(embedding)}")
                        continue
                    
                    session.run("""
                        MATCH ()-[r]->() WHERE elementId(r) = $rel_id
                        SET r.embedding = $embedding
                    """, rel_id=rel_id, embedding=embedding)
                    success_count += 1
                except Exception as e:
                    print(f"更新关系失败: {e}")
        
        return success_count
    
    def check_master_node_status(self):
        """检查Master节点embedding状态"""
        with self.neo4j_driver.session() as session:
            try:
                total = session.run("""
                    MATCH (n:__Master__)
                    WHERE n.WHU_HASNAME IS NOT NULL OR n.WHU_HASORIGINALTEXT IS NOT NULL OR n.summary IS NOT NULL
                    RETURN count(n) as cnt
                """).single()["cnt"]
                
                has_emb = session.run("""
                    MATCH (n:__Master__)
                    WHERE (n.WHU_HASNAME IS NOT NULL OR n.WHU_HASORIGINALTEXT IS NOT NULL OR n.summary IS NOT NULL)
                          AND n.embedding IS NOT NULL
                    RETURN count(n) as cnt
                """).single()["cnt"]
                
                missing = total - has_emb
                rate = (has_emb / total * 100) if total > 0 else 0
                
                print(f"Master节点: 总数{total}, 有embedding{has_emb}, 缺失{missing}, 完成率{rate:.1f}%")
                return {'total': total, 'has_embedding': has_emb, 'missing': missing, 'rate': rate}
            except Exception as e:
                print(f"检查状态失败: {e}")
                return {'total': 0, 'has_embedding': 0, 'missing': 0, 'rate': 0}
    
    def check_master_relation_status(self):
        """检查Master关系embedding状态"""
        with self.neo4j_driver.session() as session:
            try:
                total = session.run("""
                    MATCH ()-[r]->()
                    WHERE r.label_tag = '__Master__'
                          AND (r.WHU_HASNAME IS NOT NULL OR r.WHU_HASORIGINALTEXT IS NOT NULL)
                    RETURN count(r) as cnt
                """).single()["cnt"]
                
                has_emb = session.run("""
                    MATCH ()-[r]->()
                    WHERE r.label_tag = '__Master__'
                          AND (r.WHU_HASNAME IS NOT NULL OR r.WHU_HASORIGINALTEXT IS NOT NULL)
                          AND r.embedding IS NOT NULL
                    RETURN count(r) as cnt
                """).single()["cnt"]
                
                missing = total - has_emb
                rate = (has_emb / total * 100) if total > 0 else 0
                
                print(f"Master关系: 总数{total}, 有embedding{has_emb}, 缺失{missing}, 完成率{rate:.1f}%")
                return {'total': total, 'has_embedding': has_emb, 'missing': missing, 'rate': rate}
            except Exception as e:
                print(f"检查状态失败: {e}")
                return {'total': 0, 'has_embedding': 0, 'missing': 0, 'rate': 0}
    
    def process_master_nodes(self, batch_size=50):
        """处理Master节点embeddings"""
        print("\n" + "="*70)
        print("开始处理 Master 节点 Embeddings")
        print("="*70)
        
        status = self.check_master_node_status()
        if status['missing'] == 0:
            print("✓ 所有Master节点已有embedding\n")
            return True
        
        total_processed = 0
        while True:
            nodes = self.get_master_nodes_for_embedding(batch_size)
            if not nodes:
                break
            
            texts, valid_nodes = [], []
            for node in nodes:
                text = self.create_text_for_node_embedding(node)
                if text:
                    texts.append(text)
                    valid_nodes.append(node)
            
            if not texts:
                break
            
            if self.device == 'cuda':
                torch.cuda.empty_cache()
            
            embeddings = self.generate_embeddings_batch(texts)
            if not embeddings:
                break
            
            node_ids = [n['id'] for n in valid_nodes[:len(embeddings)]]
            updated = self.update_node_embeddings(node_ids, embeddings)
            total_processed += updated
            
            print(f"✓ 本批处理: {updated}/{len(valid_nodes)}")
            
            current = self.check_master_node_status()
            if current['missing'] == 0:
                print("✓ 所有Master节点处理完成!")
                break
        
        print(f"总计处理: {total_processed} 个节点\n")
        return total_processed > 0
    
    def process_master_relations(self, batch_size=50):
        """处理Master关系embeddings"""
        print("\n" + "="*70)
        print("开始处理 Master 关系 Embeddings")
        print("="*70)
        
        status = self.check_master_relation_status()
        if status['missing'] == 0:
            print("✓ 所有Master关系已有embedding\n")
            return True
        
        total_processed = 0
        while True:
            relations = self.get_master_relations_for_embedding(batch_size)
            if not relations:
                break
            
            texts, valid_rels = [], []
            for rel in relations:
                text = self.create_text_for_relation_embedding(rel)
                if text:
                    texts.append(text)
                    valid_rels.append(rel)
            
            if not texts:
                break
            
            if self.device == 'cuda':
                torch.cuda.empty_cache()
            
            embeddings = self.generate_embeddings_batch(texts)
            if not embeddings:
                break
            
            rel_ids = [r['id'] for r in valid_rels[:len(embeddings)]]
            updated = self.update_relation_embeddings(rel_ids, embeddings)
            total_processed += updated
            
            print(f"✓ 本批处理: {updated}/{len(valid_rels)}")
            
            current = self.check_master_relation_status()
            if current['missing'] == 0:
                print("✓ 所有Master关系处理完成!")
                break
        
        print(f"总计处理: {total_processed} 个关系\n")
        return total_processed > 0
    
    def create_master_index(self):
        """创建Master_embedding_index向量索引"""
        print("\n" + "="*70)
        print("创建 Master Embedding Index")
        print("="*70)
        
        with self.neo4j_driver.session() as session:
            try:
                # 检查索引是否已存在
                check_query = "SHOW INDEXES YIELD name WHERE name = 'Master_embedding_index'"
                existing = list(session.run(check_query))
                
                if existing:
                    print("索引已存在，先删除旧索引...")
                    session.run("DROP INDEX Master_embedding_index IF EXISTS")
                
                # 创建新索引
                create_query = f"""
                CREATE VECTOR INDEX Master_embedding_index IF NOT EXISTS
                FOR (n:__Master__)
                ON n.embedding
                OPTIONS {{
                    indexConfig: {{
                        `vector.dimensions`: {self.expected_dimension},
                        `vector.similarity_function`: 'cosine'
                    }}
                }}
                """
                
                session.run(create_query)
                print(f"✓ 成功创建索引: Master_embedding_index (维度: {self.expected_dimension})")
                
                # 验证索引
                verify_query = """
                SHOW INDEXES 
                YIELD name, type, labelsOrTypes, properties, state 
                WHERE name = 'Master_embedding_index'
                """
                result = session.run(verify_query).single()
                
                if result:
                    print(f"  - 类型: {result['type']}")
                    print(f"  - 标签: {result['labelsOrTypes']}")
                    print(f"  - 属性: {result['properties']}")
                    print(f"  - 状态: {result['state']}")
                else:
                    print("⚠️  索引创建成功但验证失败")
                
            except Exception as e:
                print(f"❌ 索引创建失败: {e}")
                import traceback
                traceback.print_exc()
        
        print()

def main():
    """主函数：完整的Master节点和关系embedding处理流程"""
    
    # 初始化管理器（可选：指定维度，如expected_dimension=768）
    manager = MasterEmbeddingManager()
    
    try:
        # 步骤1：处理Master节点embeddings
        manager.process_master_nodes(batch_size=50)
        
        # 步骤2：处理Master关系embeddings
        manager.process_master_relations(batch_size=50)
        
        # 步骤3：创建向量索引
        manager.create_master_index()
        
        # 最终状态报告
        print("="*70)
        print("最终状态报告")
        print("="*70)
        manager.check_master_node_status()
        manager.check_master_relation_status()
        print("="*70)
        print("✓ 所有任务完成!")
        
    except Exception as e:
        print(f"\n❌ 执行过程中发生错误: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [11]:
import logging
from utilities import return_llm_database as rldb

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class SEGAnalyzer:
    def __init__(self):
        self.driver = rldb.DatabaseManager.get_neo4j_driver(remotedatebase=False)
        self.graph_name = "SEG_In_Memory_Graph"

    def project_seg_graph(self):
        """
        利用 GDS 投影建立 SEG 内存图
        包含：所有 __Master__ 节点及其间的 MASTER_ 关系
        """
        # 注意：使用通配符匹配所有以 MASTER_ 开头的关系
        query = """
        CALL gds.graph.project.cypher(
            $graph_name,
            "MATCH (n:__Master__) RETURN id(n) AS id, labels(n) AS labels",
            "MATCH (s:__Master__)-[r]->(t:__Master__) 
             WHERE type(r) STARTS WITH 'MASTER_' 
             RETURN id(s) AS source, id(t) AS target, type(r) AS type, r.master_score AS weight",
            {validateRelationships: true}
        )
        """
        try:
            with self.driver.session() as session:
                # 如果已存在则删除，确保幂等性
                session.run(f"CALL gds.graph.drop('{self.graph_name}', false)")
                
                logger.info(f"正在建立 GDS 投影: {self.graph_name}...")
                result = session.run(query, graph_name=self.graph_name).single()
                logger.info(f"投影成功。节点数: {result['nodeCount']}, 关系数: {result['relationshipCount']}")
        except Exception as e:
            logger.error(f"投影失败: {e}")

 
        """
        统计 SEG 的全貌：节点、关系分布及权重分布
        """
        stats_query = """
        CALL gds.graph.list($graph_name) YIELD nodeCount, relationshipCount, schema
        RETURN nodeCount, relationshipCount, schema.relationships AS relDistribution
        """
        
        # 补充：统计 master_score 的分布（质量评估）
        score_query = """
        MATCH (:__Master__)-[r]->(:__Master__)
        WHERE type(r) STARTS WITH 'MASTER_'
        RETURN 
            min(r.master_score) AS min_score,
            max(r.master_score) AS max_score,
            avg(r.master_score) AS avg_score,
            stDev(r.master_score) AS std_dev
        """

        try:
            with self.driver.session() as session:
                # 1. 结构统计
                res = session.run(stats_query, graph_name=self.graph_name).single()
                print("\n" + "="*50)
                print(f"📊 SEG 内存图统计 (GDS Projection)")
                print(f"总节点数 (Master Nodes): {res['nodeCount']}")
                print(f"总关系数 (Master Relations): {res['relationshipCount']}")
                print("-" * 50)
                print(f"{'关系类型':<30} | {'出现次数'}")
                for rel_type, count in res['relDistribution'].items():
                    print(f"{rel_type:<30} | {count}")
                
                # 2. 质量统计
                score_res = session.run(score_query).single()
                print("-" * 50)
                print(f"✨ 语义质量统计 (master_score)")
                print(f"平均置信度: {score_res['avg_score']:.4f}")
                print(f"置信度区间: [{score_res['min_score']:.2f} - {score_res['max_score']:.2f}]")
                print("="*50 + "\n")

        except Exception as e:
            logger.error(f"统计获取失败: {e}")
    def get_seg_statistics(self):
        """
        保持原结构，仅修正关系分布统计逻辑
        """
        # 修改后的逻辑：通过 UNWIND 内存图中的关系类型，直接在库中统计对应数量
        stats_query = """
        CALL gds.graph.list($graph_name) YIELD nodeCount, relationshipCount, schema
        WITH nodeCount, relationshipCount, [k IN keys(schema.relationships) | k] AS relTypes
        UNWIND relTypes AS relType
        MATCH (:__Master__)-[r]->(:__Master__)
        WHERE type(r) = relType
        RETURN nodeCount, relationshipCount, relType, count(r) AS relCount
        """
        
        score_query = """
        MATCH (:__Master__)-[r]->(:__Master__)
        WHERE type(r) STARTS WITH 'MASTER_'
        RETURN 
            min(r.master_score) AS min_score,
            max(r.master_score) AS max_score,
            avg(r.master_score) AS avg_score
        """

        try:
            with self.driver.session() as session:
                # 1. 结构与关系分布统计
                results = session.run(stats_query, graph_name=self.graph_name)
                
                # 提取第一行获取总数（后续行总数一致）
                first_record = None
                rel_data = []
                for record in results:
                    if not first_record: first_record = record
                    rel_data.append((record['relType'], record['relCount']))

                if first_record:
                    print("\n" + "="*50)
                    print(f"📊 SEG 内存图统计 (GDS Projection)")
                    print(f"总节点数 (Master Nodes): {first_record['nodeCount']}")
                    print(f"总关系数 (Master Relations): {first_record['relationshipCount']}")
                    print("-" * 50)
                    print(f"{'关系类型':<35} | {'数量'}")
                    for name, count in rel_data:
                        print(f"{name:<35} | {count}")
                
                # 2. 质量统计
                score_res = session.run(score_query).single()
                print("-" * 50)
                print(f"✨ 语义质量统计 (master_score)")
                print(f"平均置信度: {score_res['avg_score']:.4f}")
                print(f"置信度区间: [{score_res['min_score']:.2f} - {score_res['max_score']:.2f}]")
                print("="*50 + "\n")

        except Exception as e:
            logger.error(f"统计获取失败: {e}")
    def close(self):
        self.driver.close()

if __name__ == "__main__":
    analyzer = SEGAnalyzer()
    try:
        analyzer.project_seg_graph()
        analyzer.get_seg_statistics()
    finally:
        analyzer.close()

2025-12-16 15:36:47,014 - INFO - 正在建立 GDS 投影: SEG_In_Memory_Graph...
2025-12-16 15:36:47,045 - WARNING - Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('schema' returned by 'gds.graph.drop' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL gds.graph.drop('SEG_In_Memory_Graph', false)"
2025-12-16 15:36:47,301 - WARNING - Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('gds.graph.project.cypher' has been replaced by 'gds.graph.project Cypher projection as an aggregation function')} {position: line: 2, col


📊 SEG 内存图统计 (GDS Projection)
总节点数 (Master Nodes): 4310
总关系数 (Master Relations): 1744
--------------------------------------------------
关系类型                           | 出现次数
MASTER_mp_supports             | {'weight': 'Float (DefaultValue(NaN), TRANSIENT, Aggregation.NONE)'}
MASTER_prov_generated          | {'weight': 'Float (DefaultValue(NaN), TRANSIENT, Aggregation.NONE)'}
MASTER_mp_challenges           | {'weight': 'Float (DefaultValue(NaN), TRANSIENT, Aggregation.NONE)'}
MASTER_whu_declareUsed         | {'weight': 'Float (DefaultValue(NaN), TRANSIENT, Aggregation.NONE)'}
MASTER_whu_fellow              | {'weight': 'Float (DefaultValue(NaN), TRANSIENT, Aggregation.NONE)'}
MASTER_p_plan_isOutputVarOf    | {'weight': 'Float (DefaultValue(NaN), TRANSIENT, Aggregation.NONE)'}
MASTER_whu_hasGoal             | {'weight': 'Float (DefaultValue(NaN), TRANSIENT, Aggregation.NONE)'}
MASTER_prov_wasInformedBy      | {'weight': 'Float (DefaultValue(NaN), TRANSIENT, Aggregation.NONE)'}
MASTER_pr

2025-12-16 15:36:47,547 - WARNING - Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('schema' returned by 'gds.graph.list' is deprecated.)} {position: line: 2, column: 78, offset: 78} for query: '\n        CALL gds.graph.list($graph_name) YIELD nodeCount, relationshipCount, schema\n        WITH nodeCount, relationshipCount, [k IN keys(schema.relationships) | k] AS relTypes\n        UNWIND relTypes AS relType\n        MATCH (:__Master__)-[r]->(:__Master__)\n        WHERE type(r) = relType\n        RETURN nodeCount, relationshipCount, relType, count(r) AS relCount\n        '



📊 SEG 内存图统计 (GDS Projection)
总节点数 (Master Nodes): 4310
总关系数 (Master Relations): 1744
--------------------------------------------------
关系类型                                | 数量
MASTER_prov_used                    | 26
MASTER_whu_hasActivity              | 183
MASTER_whu_fellow                   | 90
MASTER_p_plan_isOutputVarOf         | 29
MASTER_prov_generated               | 84
MASTER_prov_wasInformedBy           | 249
MASTER_mp_challenges                | 297
MASTER_whu_declareUsed              | 97
MASTER_whu_hasGoal                  | 107
MASTER_mp_supports                  | 582
--------------------------------------------------
✨ 语义质量统计 (master_score)
平均置信度: 0.7997
置信度区间: [0.50 - 0.95]



## C16(废弃) 直接实现基于embedding相似度的合并，不建立master entity

#### A）第一步：实现返回相似度的id，而不是name，为合并提供参数：

In [ ]:
from neo4j import GraphDatabase
import utilities.return_llm_database as db_manager
import pprint as pp

#manager = utilities.return_llm_database.DatabaseManager()
#llm, embed_model, neo4j_driver = manager.get_components()
neo4j_driver=db_manager.DatabaseManager.get_neo4j_driver(remotedatebase=False)
similarity_threshold = 0.90

CYPHER_GROUPS = """
MATCH (e:__Entity__)
WHERE e.embedding IS NOT NULL
  AND coalesce(e.WHU_HASNAME, e.WHU_HASORIGINALTEXT, '') <> ''
CALL {
  WITH e
  CALL db.index.vector.queryNodes('entity_embedding_index', 10, e.embedding)
  YIELD node, score
  WITH e, node, score
  WHERE node.embedding IS NOT NULL
    AND score > toFloat($cutoff)
    AND labels(e) = labels(node)
    AND coalesce(node.WHU_HASNAME, node.WHU_HASORIGINALTEXT, '') <> ''
  WITH node, score
  ORDER BY coalesce(node.WHU_HASNAME, node.WHU_HASORIGINALTEXT, '')
  RETURN collect(node) AS nodes
}
WITH DISTINCT nodes
WHERE size(nodes) > 1

// 把候选簇投影为 “id 列表”
WITH collect([n IN nodes | elementId(n)]) AS results

UNWIND range(0, size(results)-1, 1) AS index
WITH results, index, results[index] AS result
WITH apoc.coll.sort(
  reduce(
    acc = result,
    index2 IN range(0, size(results)-1, 1) |
      CASE
        WHEN index <> index2 AND size(apoc.coll.intersection(acc, results[index2])) > 0
          THEN apoc.coll.union(acc, results[index2])
        ELSE acc
      END
  )
) AS combinedIds

WITH DISTINCT combinedIds

// 额外去重：去掉被完全包含的集合
WITH collect(combinedIds) AS allCombinedResults
UNWIND range(0, size(allCombinedResults)-1, 1) AS combinedResultIndex
WITH allCombinedResults[combinedResultIndex] AS combinedIds, combinedResultIndex, allCombinedResults
WHERE NOT any(x IN range(0, size(allCombinedResults)-1, 1)
  WHERE x <> combinedResultIndex
    AND apoc.coll.containsAll(allCombinedResults[x], combinedIds)
)

RETURN combinedIds
"""

records, _, _ = neo4j_driver.execute_query(
    CYPHER_GROUPS,
    {"cutoff": similarity_threshold},
)
groups = [rec["combinedIds"] for rec in records]
print(f"candidate groups: {len(groups)}")
for g in groups[:5]:
    pp.pprint(g)


#### B) 第二步：把每个簇 [a,b,c] 合并为新节点 d（保留所有关系）

我们用 apoc.create.node 创建 d，再用 apoc.refactor.mergeNodes([d]+nodes,{mergeRels:true,properties:'discard'}) 合并，保证 d 的属性为我们指定的“规范结果”，并把所有关系并到 d 上。

WHU_HASORIGINALTEXTS：直接存字符串数组。

若你确实要“字典（map）”，请把它序列化为 JSON 字符串存到 WHU_ORIG_MAP_JSON。

embedding：默认复制候选里“名称最长”的一个的 embedding；如果你想重算，把 RECOMPUTE_EMBEDDING 设为 True 并提供合成文本。

In [ ]:
import json

# 可选：是否为 d 重算 embedding（比“复制一个”更干净）
RECOMPUTE_EMBEDDING = False  # True 则用 embed_model 对合成文本向量化

MERGE_QUERY = """
// 参数：$ids(list<string>)  $name(string)  $aliases(list<string>)  $origTexts(list<string>)  $embedding(list<float>)  $sourceIds(list<string>)  $labels(list<string>)  $origMapJson(string or null)

MATCH (n) WHERE elementId(n) IN $ids
WITH collect(n) AS ns, $labels AS lbls

// 创建 d，并写入我们想要的规范属性
CALL apoc.create.node(lbls, {
  WHU_HASNAME: $name,
  WHU_ALIASES: $aliases,
  WHU_HASORIGINALTEXTS: $origTexts,
  sourceElementIds: $sourceIds,
  embedding: $embedding,
  WHU_ORIG_MAP_JSON: $origMapJson
}) YIELD node AS d

// 把 ns 全部并进 d，保留 d 的属性，合并所有关系
CALL apoc.refactor.mergeNodes([d] + ns, {mergeRels:true, properties:'discard'}) YIELD node
RETURN elementId(node) AS mergedId
"""

def choose_labels_for_group(ids):
    # 取第一个节点的标签作为 d 的标签（你也可以固定用 __Entity__）
    q = "MATCH (n) WHERE elementId(n) IN $ids RETURN labels(n) AS lbl LIMIT 1"
    recs, _, _ = neo4j_driver.execute_query(q, {"ids": ids})
    return recs[0]["lbl"] if recs else ["__Entity__"]

def fetch_group_payload(ids):
    # 取名称/原文/embedding/可选 chunk id（若有）
    q = """
    MATCH (n) WHERE elementId(n) IN $ids
    RETURN
      elementId(n)            AS id,
      coalesce(n.WHU_HASNAME, n.WHU_HASORIGINALTEXT, '') AS name_like,
      n.WHU_HASNAME           AS name,
      n.WHU_HASORIGINALTEXT   AS orig_text,
      n.embedding             AS emb,
      n.WHU_CHUNK_IDS         AS chunk_ids   // 若没有这个属性没关系，返回 None
    """
    recs, _, _ = neo4j_driver.execute_query(q, {"ids": ids})
    names = []
    orig_texts = []
    embeddings = []
    chunk_map = {}  # {id: [chunkIds]}
    for r in recs:
        if r["name"]:
            names.append(r["name"])
        if r["orig_text"]:
            orig_texts.append(r["orig_text"])
        if r["emb"]:
            embeddings.append((r["id"], r["emb"], len(r["name_like"] or "")))
        if r["chunk_ids"]:
            chunk_map[r["id"]] = r["chunk_ids"]
    return names, orig_texts, embeddings, chunk_map

def canonicalize_name(names):
    # 你要求“投入 LLM 综合”，这里给一个兜底：无 LLM 就选最长的；有 llm 可替换
    if not names:
        return ""
    names_uniq = sorted(set([n.strip() for n in names if n and n.strip()]), key=len, reverse=True)
    # 如需 LLM：把 names_uniq 丢给 llm 让它输出一个“规范名”，此处给最快兜底
    return names_uniq[0], names_uniq

def pick_embedding(embeddings, recompute_text=None):
    # embeddings: [(id, vec, name_len)]
    if RECOMPUTE_EMBEDDING and recompute_text:
        # 用你现有的 embed_model 计算
        vec = embed_model.embed_query(recompute_text)
        return vec
    # 否则：选“name_like 最长”的那个
    if not embeddings:
        return None
    best = sorted(embeddings, key=lambda t: t[2], reverse=True)[0]
    return best[1]

seen = set()  # 已处理过的节点，避免重复合并

merged_count = 0
for ids in groups:
    # 跳过已处理过的节点
    if any(i in seen for i in ids):
        continue

    labels_for_d = choose_labels_for_group(ids)
    names, orig_texts, embeddings, chunk_map = fetch_group_payload(ids)
    canon_name, aliases = canonicalize_name(names)

    # 你也可以组装一个“合成文本”用于重算 embedding（如：规范名 + 前几条原文摘要）
    recompute_text = canon_name  # 简化：只用规范名（可拼接 orig_texts[:1] 等）
    emb = pick_embedding(embeddings, recompute_text=recompute_text)

    orig_map_json = json.dumps(chunk_map, ensure_ascii=False) if chunk_map else None

    params = {
        "ids": ids,
        "name": canon_name,
        "aliases": sorted(set(aliases)),
        "origTexts": orig_texts[:50],          # 防止特别长；根据需要截断
        "sourceIds": ids,
        "embedding": emb,
        "labels": labels_for_d,
        "origMapJson": orig_map_json,
    }

    recs, _, _ = neo4j_driver.execute_query(MERGE_QUERY, params)
    merged_id = recs[0]["mergedId"] if recs else None

    seen.update(ids)
    merged_count += 1
   #print(f"[merged] {ids}  ->  {merged_id}")

print(f"done: merged groups = {merged_count}")


# pipeline code block D 基于Agent的实体对齐